# Who Wants to Be a PoliMillionaire? — NLP 2025/26 Group Assignment

## Group members
- Jiaxin Yang — jiaxin.yang@mail.polimi.it
- Runjie (Simone) Dai — runjie.dai@mail.polimi.it

## Video
- Presentation video (≤5 min): https://youtu.be/Ub6XI_UnmH4?is=sGyjGIwOUgvoBZf9

## Statement on coding assistants
We used a coding assistant (Claude Code) to help write, refactor and document parts of the
implementation. The system design, the experiments, and the prompt / RAG / routing strategies and
all analysis are our own; the assignment was **not** handed to an LLM to complete.

---

## What this notebook is
A self-explanatory consolidation of our whole pipeline and study, in narrative order:

| § | Section | Rubric question it answers |
|---|---|---|
| 1 | Setup | — (clone branch `main`, install, paths, game client) |
| 2 | Baseline single-model QA | 30s feasibility · per-topic strengths · overconfidence · failure modes |
| 3 | Prompt engineering | best prompt (zero / few / CoT) · prompt sensitivity |
| 4 | Adaptive prompt routing | adaptive prompting · *when does reasoning help vs hurt* · failure taxonomy |
| 5 | Tools + ensemble voting | calculator lift · self-consistency / ensemble reliability (agentic AI) |
| 6 | RAG (live web, raw content only) | RAG lift · RAG-vs-no-RAG ablation |
| 7 | Model comparison | model A/B · size vs quality |
| 8 | Live play — the real game | the leaderboard result (the actual test) |
| 9 | Conclusions | every investigation question → our finding |

> **How to read / reproduce.** Outputs are **preserved** from each experiment's own Colab run — the
> offline studies (§2–§7) are not meant to be re-run end-to-end in one pass (each loads the 7B model
> with its own config). To reproduce a single section: run **§1 Setup**, then that section's cells.
> The **live leaderboard run (§8)** is the only section we refresh for the final submission.
>
> Rules respected throughout: open-weight model run **locally** on Colab (`Qwen/Qwen2.5-7B-Instruct`,
> 4-bit), ≤30 s/question, and RAG uses **raw retrieved content only** (never generated answers).
> Code comments are in Yoda style (an assignment requirement).

---
# §1 · Setup — clone (branch `main`), install deps, paths, game client

One canonical setup for the whole notebook. Pulls the code from GitHub (branch **`main`**), installs
the inference stack + headless Chromium (for the live-News body fetch), and puts `src` + the provided
`millionaire_client` on the path. Run this once.

In [ ]:
# Auto-reload edited src modules on every cell run -- so after a `git pull` the newest code lands without a
# manual importlib.reload or a restart. (Re-run the cell that USES the code, e.g. code-wire.)
# Colab's IPython ships an autoreload that does `from imp import reload`, and `imp` is GONE in Python 3.12 --
# so a tiny `imp` shim (reload only) we install first, then load the extension. BEST-EFFORT: any failure
# caught, so the cell never stalls (fall back: after a src pull, Runtime > Restart to pick changes up).
try:
    import sys as _sys, types as _types, importlib as _importlib
    if 'imp' not in _sys.modules:
        _imp = _types.ModuleType('imp')
        _imp.reload = _importlib.reload          # the one thing the old autoreload.py wants from `imp`.
        _sys.modules['imp'] = _imp
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload: ON (src edits hot-reload on cell re-run)')
except Exception as _e:
    print(f'autoreload OFF ({type(_e).__name__}: {_e}) -- after a src pull, Runtime > Restart to pick changes up.')

import os, sys

REPO_URL = 'https://github.com/SleepyEveryD/NLPDelivery.git'
REPO_ROOT = '/content/NLP'
BRANCH = 'main'
if not os.path.exists(REPO_ROOT):
  !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
  # Already cloned -> HARD-SYNC to the latest pushed branch (fetch + force-reset to origin/{BRANCH}).
  # Tracked files are overwritten to match remote; UNTRACKED run outputs are KEPT (experiments/runs/* is
  # gitignored). NOTE: `git pull` updates the FILES on disk -- it does NOT refresh THIS notebook's cells.
  !cd {REPO_ROOT} && git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH}

!cd {REPO_ROOT} && echo "on branch:" $(git rev-parse --abbrev-ref HEAD) "@" $(git --no-pager log -1 --oneline)

SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
  if p not in sys.path:
    sys.path.insert(0, p)
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

# --- Persistent model cache on Google Drive ------------------------------------------------------
# The Qwen 7B weights (~15GB) download ONCE to Drive, then every notebook/session reuses them --
# no repeat downloads. Set BEFORE transformers is imported, this must be. BEST-EFFORT: outside Colab,
# or if the Drive prompt you decline, fall back to the ephemeral cache (re-downloads, but never errors).
try:
    from google.colab import drive as _drive
    if not os.path.isdir('/content/gdrive/MyDrive'):
        _drive.mount('/content/gdrive')
    _hf = '/content/gdrive/MyDrive/hf_cache'
    os.makedirs(_hf, exist_ok=True)
    os.environ['HF_HOME'] = _hf
    print('HF cache ->', _hf, '(model downloads once, reused after)')
except Exception as _e:
    print(f'Drive cache off ({type(_e).__name__}) -- ephemeral cache, the model re-downloads each session.')

In [81]:
# The inference stack + the client's `requests`, install we do (light it stays). `-U` kept (Colab a stale
# bitsandbytes preinstalls); pandas/requests PINNED to Colab's versions; matplotlib NOT upgraded (bare `-U` breaks google-colab/cudf, and upgrading matplotlib mid-session breaks the already-imported backend_bases). Colab ships matplotlib; requirements.txt covers local.
!pip install -q -U 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.46.1' sentencepiece einops pyyaml 'pandas==2.2.2' 'requests==2.32.4'
print('Installed, the dependencies are.')

Installed, the dependencies are.


In [82]:
# Headless Chromium -- the live-NEWS body fetch it powers (configs/live.yaml: news_body_mode "browser").
# The relevance gate now ROUTES off-topic Guardian results here, so the browser matters MORE for News.
# Skip this only if you set retrieval.news_body_mode: "off".
!pip install -q playwright
!playwright install chromium
!playwright install-deps

# ARMED? Probe inside a WORKER THREAD (its own event loop) -- EXACTLY how the real fetcher runs
# (src/retrieval/browser_fetch.py). A bare sync_playwright() in the cell would hit Jupyter/Colab's
# asyncio loop and FALSELY report "not ready" ("Sync API inside asyncio loop"), even though live-News
# body fetch works fine. NOT ready -> News falls back to HEADLINES only (crash-safe).
import threading
_probe = {}
def _probe_browser():
    try:
        from playwright.sync_api import sync_playwright
        with sync_playwright() as _p:
            _b = _p.chromium.launch(headless=True); _b.close()
        _probe['ok'] = True
    except Exception as _e:
        _probe['err'] = f'{type(_e).__name__}: {_e}'
_t = threading.Thread(target=_probe_browser); _t.start(); _t.join(timeout=120)
if _probe.get('ok'):
    print('headless Chromium: READY -- live-News body fetch armed.')
else:
    print(f"headless Chromium NOT ready ({_probe.get('err', 'launch timed out')})")
    print('   -> News will use HEADLINES only. Re-run this cell, or set retrieval.news_body_mode: "off".')

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entr

---
# §2 · Baseline single-model QA  *(Phase 1 — the MVP)*

The first answering pipeline: load Qwen2.5-7B 4-bit once, wire the `QAPipeline`, and benchmark zero-shot
over the dev set. This section establishes **30 s feasibility**, **per-topic strengths**,
**overconfidence**, and the dominant **failure modes** — the questions the rubric asks first.

## 2 · Load the run config
Into every run logged it is (D-007) — reproducible, the science stays.

In [83]:
from config import RunConfig

# The base config, from YAML we load.
config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'base.yaml'))
print('run_id:', config.run_id)
print('model:', config.model.name, '|', config.model.quantization, '|', config.model.dtype)
print('latency budget:', config.latency_budget_s, 's')
print('prompt strategy:', config.prompt_strategy)

run_id: baseline
model: Qwen/Qwen2.5-7B-Instruct | 4bit | bfloat16
latency budget: 30.0 s
prompt strategy: zero_shot_v1


## 3 · Load + warm up the model
On a T4, ~1–2 minutes the 4-bit load takes (a one-time cost). Warmup pays the first-call latency. here check that the model is working
**before** any question is timed — so the 30s budget, a cold start never eats it.

In [ ]:
import time
#TransformersEngine is used to handle transformers
from inference.engine import TransformersEngine

t0 = time.perf_counter()
# Free any model left in VRAM by a previous run -- re-running this cell must NOT stack a 2nd model on
# the T4 (THAT is the CUDA OutOfMemory). Del + empty_cache BEFORE the new one we allocate, so the old
# 4-bit model's ~6GB is released first (Python loads the RHS before rebinding `engine`, else they stack).
import gc as _gc
try:
    import torch as _torch
    for _v in ('engine', 'engine_math'):
        if _v in globals():
            del globals()[_v]
    _gc.collect()
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()
except Exception as _e:
    print('VRAM pre-free skipped:', type(_e).__name__)

engine = TransformersEngine(
    model_name=config.model.name,
    quantization=config.model.quantization,
    dtype=config.model.dtype,
)
load_s = time.perf_counter() - t0
print(f'Model loaded in {load_s:.1f}s')

# Warm up -- the first-call kernels, compiled before any timed question they are.
t0 = time.perf_counter()
engine.warmup()
warm_s = time.perf_counter() - t0
print(f'Warmup in {warm_s:.1f}s')

# A smoke generation -- the answering path works, confirm we do.
t0 = time.perf_counter()
out = engine.generate('Reply with the single letter B and nothing else.', max_new_tokens=5)
gen_s = time.perf_counter() - t0
print(f'Smoke generate in {gen_s:.2f}s -> {out!r}')
print(f'tokens_in={engine.last_tokens_in}, tokens_out={engine.last_tokens_out}')


## 4 · Wire the pipeline + a single-question demo
Dependency injection (D-006): engine, classifier and prompt builder, injected they are.
RAG and tools, off for the baseline they stay (Phase 4 and Phase 3 turn them on).

In [ ]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline
from evaluation.dataset import load_questions

# The collaborators, injected into the pipeline they are (D-006).
classifier = QuestionClassifier()
prompt_builder = PromptBuilder(strategy=config.prompt_strategy)
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=prompt_builder,
    classifier=classifier,
    retriever=None,   # Off for the baseline -- Phase 4 turns it on.
    tools=None,       # Off for the baseline -- Phase 3 turns it on.
    latency_budget_s=config.latency_budget_s,
)

# The dev question set, from disk loaded it is.
questions = load_questions(os.path.join(REPO_ROOT, 'data', 'dev_questions.jsonl'))
print(f'Loaded {len(questions)} dev questions.')

# One question, end to end through the pipeline we run -- the moving parts, inspect them we do.
demo_q = questions[0]
enriched = classifier.classify(demo_q)
print('--- The built prompt ---')
print(prompt_builder.build(enriched))
print()
pred = pipeline.answer(demo_q)
print('--- The prediction ---')
print('raw_output:', repr(pred.raw_output))
print('parsed answer:', pred.answer, '| gold:', demo_q.gold, '| confidence:', pred.confidence)
print('latency_s:', round(pred.latency_s, 2), '| tokens_out:', pred.tokens_out)

### 🔬 Core implementation: the seven-stage `QAPipeline.answer()` orchestration

Above we `import`ed `QAPipeline`. Its full logic lives in `src/agent/pipeline.py`; here we show the
**core orchestration** directly, so the whole system is easy to follow: one question in -> one
`Prediction` out. Seven stages run in order, every collaborator is dependency-injected (dormant when
`None`), and the whole thing is crash-safe (in live mode we must ALWAYS be able to submit an answer).

```python
def answer(self, question: Question) -> Prediction:
    guard = LatencyGuard(self.latency_budget_s)   # 30s budget, timed end-to-end
    try:
        # 1) classify -- tag the question with topic / type / language
        if self.classifier:
            question = self.classifier.classify(question)

        # 2) solver short-circuit (Maths only) -- a deterministic hit answers directly, skips the LLM
        if self.solver is not None:
            sol = self.solver(question)
            if sol is not None:
                letter, evidence = sol
                return Prediction(qid=question.qid, answer=letter, confidence=1.0,
                                  tool_used="math_solver", ...)

        # 3) retrieve -- only if a retriever exists AND the classifier says evidence is needed
        #    (RAG, RAW documents, never an LLM answer)
        if self.retriever and (not self.classifier or self.classifier.needs_retrieval(question)):
            docs = self.retriever.retrieve(question)
            retrieval_used = True

        # 4) prompt -- build the user-turn from the question + optional evidence
        prompt = self.prompt_builder.build(question, docs)

        # 5) generate -- n=1 single greedy pass; n>1 runs self-consistency (vote over N sampled CoT chains)
        if self.self_consistency_n > 1:
            ans, conf, raw = self._self_consistency_answer(prompt, question, guard)
        else:
            raw = self._gen(prompt)

        # 6) tool -- the calculator as a "verifier": only on arithmetic Qs, no retrieval, not self-consistency
        if (self.self_consistency_n <= 1 and self.tools and self.classifier
                and not retrieval_used and self.classifier.needs_calculator(question)):
            tool_raw, used = self._run_calculator_tool(question, guard)
            if tool_raw is not None:
                raw, tool_used = tool_raw, used

        # 7) parse -- robustly turn the model text into (answer, confidence)
        if self.self_consistency_n <= 1:
            ans, conf = QAPipeline.parse_answer(raw, question)

    except Exception as e:
        # any crash -> fall back to the first option; a live game never goes silent
        return Prediction(..., answer=fallback_ans, confidence=0.0, error=str(e))

    return Prediction(qid=question.qid, answer=ans, confidence=conf, ...)
```

**Parser robustness**: `parse_answer()` extracts the option letter from the model's free text with a
7-tier pattern ladder -- `Answer: B` -> `(C)` -> `Option D` -> `B)` -> `B Rome` (echoes the option
text) -> a whole-line single letter -> any isolated letter, confidence decreasing down the ladder; if
nothing matches it falls back to the first option (live must have an answer). Full implementation in
`src/agent/pipeline.py`.


## 5 · Run the baseline benchmark
Over all dev questions the pipeline runs; one `EvalRecord` per question, to JSONL it is logged (D-007).
Crash-safe it is — each line flushed immediately, so a Colab disconnect nothing it loses.

In [ ]:
from evaluation.runner import BenchmarkRunner

# Into the repo the run logs we write.
runner = BenchmarkRunner(
    pipeline=pipeline,
    config=config,
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
)
run_path = runner.run(questions)
print('Run written to:', run_path)

## 6 · Results — accuracy + latency
From the JSONL log all of it derives — re-analyse without re-running the model, we can.

In [ ]:
from evaluation.metrics import load_runs, accuracy_by, latency_summary

# The logged run, back into a DataFrame we read.
df = load_runs([run_path])

# Overall accuracy, the headline number it is.
known = df[df['correct'].notna()]
overall = known['correct'].astype(float).mean()
print(f'Overall accuracy: {overall:.1%}  (n={len(known)})')
print()
print('--- Accuracy by topic ---')
print(accuracy_by(df, 'topic').to_string(index=False))
print()
print('--- Accuracy by level ---')
print(accuracy_by(df, 'level').to_string(index=False))
print()
print('--- Latency summary (seconds) ---')
for k, v in latency_summary(df).items():
    print(f'{k}: {v}')

In [ ]:
import matplotlib.pyplot as plt

# Accuracy by topic, a bar chart it becomes -- per-topic strengths, the rubric asks for them.
topic_acc = accuracy_by(df, 'topic').sort_values('accuracy')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(topic_acc['topic'], topic_acc['accuracy'], color='steelblue')
ax[0].set_xlim(0, 1)
ax[0].set_xlabel('accuracy')
ax[0].set_title('Baseline accuracy by topic')

# Latency per question, a histogram it forms -- the 30s wall, a red line marks it.
ax[1].hist(df['latency_s'].dropna(), bins=12, color='darkorange', edgecolor='black')
ax[1].axvline(30.0, color='red', linestyle='--', label='30s budget')
ax[1].set_xlabel('latency (s)')
ax[1].set_ylabel('questions')
ax[1].set_title('Per-question latency')
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
miss = df[(df['correct'] == False)][['qid', 'topic', 'question_text', 'predicted_answer', 'gold_answer', 'confidence',
  'raw_output']]
import pandas as pd
pd.set_option('display.max_colwidth', None)
print(miss.to_string(index=False))

## 30s feasibility
The baseline pipeline runs comfortably under the 30s limit on a T4 GPU.

Latency summary:
- Median latency: ~0.91s
- p95 latency: ~1.01s
- Max latency: ~1.05s
- Budget violation rate: 0%

This leaves a very large safety margin under the 30-second game constraint.

---

## Per-topic strengths
Strongest topics at baseline:
- Ancient History and Politics — 100%
- Entertainment — 100%
- News — 100%
- Philosophy and Psychology — 100%

Weakest topics:
- Maths — 50% accuracy (worst performing topic)
- Science and Nature — 75% accuracy

This suggests the zero-shot prompting setup performs well on factual recall and general knowledge, but struggles more with reasoning-heavy or calculation-based questions.

---

## Overconfidence
Yes — there is evidence of overconfidence.

Example:
- The model answered the moons question incorrectly (`A` instead of the correct answer `B`)
- Reported confidence was still `1.0`

This indicates the baseline system is poorly calibrated: it can produce highly confident but incorrect answers.

---

## Failure modes
The observed failures appear to be mainly real knowledge or reasoning gaps rather than parser failures.

Evidence:
- Raw outputs were generally clean single-letter responses such as `'A'`
- The parser extracted answers successfully
- No obvious formatting or chatter issues appeared in the baseline run

The main failure causes are likely:
- Weak mathematical reasoning
- Incorrect factual recall
- Limited reasoning ability under zero-shot prompting

No major parser breakdowns were observed.

---

## Next steps
Next, we should test:
- `few_shot_v1`
- `cot_v1` (chain-of-thought prompting)

These prompting strategies may improve reasoning accuracy, especially for Maths and Science questions, while still remaining comfortably within the 30-second latency budget.

---
# §3 · Prompt engineering  *(Phase 2)*

Three prompt strategies head-to-head on the dev set — **zero-shot**, **few-shot**, **chain-of-thought** —
plus `cot_v2`, a CoT variant driven by two real failure cases (option-matching slips on Maths). Which
prompt is truly best, and how sensitive is the model to phrasing?

# 02 · Prompt engineering  (Phase 2)

**Who Wants to Be a PoliMillionaire?** — three prompt strategies, head to head this section runs.

The baseline (§2) scored **87%**, with the misses concentrated in **Maths** (arithmetic
errors) and one **Science** knowledge-cutoff trap. The central question here:

> Does **chain-of-thought** (think step by step) rescue the arithmetic on its own — or is a
> deterministic **calculator tool** (Phase 3) actually required?

Same engine, same dev set, same pipeline seam — **only the prompt strategy changes**:
`zero_shot_v1` · `few_shot_v1` · `cot_v1`. The model, loaded ONCE it is; three benchmarks then run.

> **GPU needed.** Runtime ▸ Change runtime type ▸ T4 GPU, select you must.

## 2 · Load the model once + the dev set
The 4-bit load (~1–2 min), paid a single time it is -- across all three strategies, reused the engine stays.

In [ ]:
import time
from config import RunConfig
from inference.engine import TransformersEngine
from classify.classifier import QuestionClassifier
from evaluation.dataset import load_questions

config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'base.yaml'))

# Once, the model we load.
t0 = time.perf_counter()
# Free any model left in VRAM by a previous run -- re-running this cell must NOT stack a 2nd model on
# the T4 (THAT is the CUDA OutOfMemory). Del + empty_cache BEFORE the new one we allocate, so the old
# 4-bit model's ~6GB is released first (Python loads the RHS before rebinding `engine`, else they stack).
import gc as _gc
try:
    import torch as _torch
    for _v in ('engine', 'engine_math'):
        if _v in globals():
            del globals()[_v]
    _gc.collect()
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()
except Exception as _e:
    print('VRAM pre-free skipped:', type(_e).__name__)

engine = TransformersEngine(
    model_name=config.model.name,
    quantization=config.model.quantization,
    dtype=config.model.dtype,
)
engine.warmup()
print(f'Model ready in {time.perf_counter() - t0:.1f}s')

# The collaborators that DO NOT change across strategies, here once we build.
classifier = QuestionClassifier()
questions = load_questions(os.path.join(REPO_ROOT, 'data', 'dev_questions.jsonl'))
print(f'Loaded {len(questions)} dev questions.')


## 3 · Run all three strategies
Per strategy, a fresh `run_id` and `PromptBuilder` we make -- everything else, identical it stays.
One logged run per strategy results.

### 🔬 Core implementation: the prompt-strategy library + `cot_v2` (driven by two real failures)

The three strategies (zero-shot / few-shot / CoT) are all registered in `_REGISTRY` in
`src/prompting/builder.py`; `PromptBuilder(strategy)` looks one up by name. Our most important strategy
is **`cot_v2`**, forced out by two **real live failures**:

- **run #7 (option-matching slip)**: on a t-test question the model's reasoning was actually correct
  (df=17, +/-2.110 = the content of option C), yet it wrote `Answer: B` -- B/C share the same conclusion
  ("do not reject"), and the model matched only the conclusion without re-checking the numbers buried in
  the options.
- **run #9 (truncation loss)**: the set-up was correct, but it wrote ~5 paragraphs of LaTeX and **never
  reached the `Answer:` line** at the 256-token cap, so the parser could only blindly pick A. At ~11
  tok/s, more tokens just time out -- the fix is **fewer tokens to the answer**: cap the steps, ban
  LaTeX, force an explicit `Answer:` line.

`cot_v2`'s core instruction for MCQs (from `_cot_v2`):

```python
"Solve in AT MOST 3 very short steps. Plain numbers ONLY -- NO LaTeX, no \\frac, "
"no \\mu/\\sigma, no $...$; write 'mu'/'sigma' as words and keep each step under ~12 "
"words. When two options share the same conclusion, pick the one whose numbers (values, "
"signs, degrees of freedom) match your result EXACTLY -- not just the conclusion. You "
"MUST end on a new line with 'Answer: X' (X = A, B, C, or D) -- always reach that line."
```

These three constraints map exactly onto the two bugs above: **step-cap + no-LaTeX** fixes truncation,
**check every number, not just the conclusion** fixes the option-matching slip, and the **forced
`Answer:` line** backstops the parser. The cells below compare the three strategies on the dev set;
Maths is the topic where they differ most.


In [ ]:
from dataclasses import replace
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline
from evaluation.runner import BenchmarkRunner

strategies = ['zero_shot_v1', 'few_shot_v1', 'cot_v1']
run_paths = []
for s in strategies:
    # A fresh config per strategy -- a distinct run_id, its own log dir gives it.
    cfg = replace(config, run_id='phase2_' + s, prompt_strategy=s)
    pipe = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=s),
        classifier=classifier,
        latency_budget_s=config.latency_budget_s,
    )
    runner = BenchmarkRunner(pipe, cfg, log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'))
    print('=== strategy:', s, '===')
    run_paths.append(runner.run(questions))
print()
print('Runs:', run_paths)

## 4 · Compare — overall, per-topic, and Maths in focus
The CoT-vs-tool question, the Maths column answers it.

In [ ]:
import pandas as pd
from evaluation.metrics import load_runs, accuracy_by

# All three runs, into one DataFrame we read.
df = load_runs(run_paths)

print('--- Overall accuracy by strategy ---')
print(accuracy_by(df, 'prompt_strategy').to_string(index=False))
print()

# Maths only -- does any strategy fix the arithmetic, see we do.
maths = df[df['topic'] == 'Maths']
print('--- MATHS accuracy by strategy ---')
print(accuracy_by(maths, 'prompt_strategy').to_string(index=False))
print()

# Topic x strategy, the full picture it paints.
known = df[df['correct'].notna()].copy()
known['correct'] = known['correct'].astype(float)
pivot = known.pivot_table(index='topic', columns='prompt_strategy', values='correct', aggfunc='mean')
print('--- Accuracy: topic x strategy ---')
print(pivot.round(2).to_string())
print()

# Latency + output length -- CoT costs tokens, confirm we do.
print('--- Mean latency (s) and mean tokens_out by strategy ---')
print(df.groupby('prompt_strategy')[['latency_s', 'tokens_out']].mean().round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt

overall = accuracy_by(df, 'prompt_strategy').set_index('prompt_strategy')['accuracy']
maths_acc = accuracy_by(maths, 'prompt_strategy').set_index('prompt_strategy')['accuracy']
lat = df.groupby('prompt_strategy')['latency_s'].mean()
order = ['zero_shot_v1', 'few_shot_v1', 'cot_v1']

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(order, [overall.get(s, 0) for s in order], color='steelblue')
ax[0].set_ylim(0, 1); ax[0].set_title('Overall accuracy'); ax[0].set_ylabel('accuracy')
ax[1].bar(order, [maths_acc.get(s, 0) for s in order], color='seagreen')
ax[1].set_ylim(0, 1); ax[1].set_title('MATHS accuracy (the key question)')
ax[2].bar(order, [lat.get(s, 0) for s in order], color='darkorange')
ax[2].axhline(30.0, color='red', linestyle='--', label='30s budget')
ax[2].set_title('Mean latency (s)'); ax[2].legend()
for a in ax:
    a.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
miss = df[(df['correct'] == False)][['qid', 'topic', 'question_text', 'predicted_answer', 'gold_answer', 'confidence',
  'raw_output']]
import pandas as pd
pd.set_option('display.max_colwidth', None)
print(miss.to_string(index=False))

## 5 · Read-off

Reading the comparison above (and confirmed by the deeper study in §4):

- **No single best strategy.** Few-shot is strongest on factual recall; chain-of-thought helps the
  reasoning-heavy topics. The biggest movement is on **Maths**, the weakest baseline topic (0.50).
- **CoT helps Maths reasoning, but the win is fragile.** Our `cot_v2` variant fixes the two real failure
  modes above (option-matching slips + LaTeX truncation). §4 shows *which* question shapes CoT helps vs
  hurts — recall questions regress under forced reasoning, which is why we route per shape rather than
  force CoT everywhere.
- **A deterministic calculator is not required.** Once `cot_v2` lands the arithmetic the calculator adds
  no net accuracy and (at n=1) can clobber correct chains — so live Maths runs it **off** (see §5 / §8).
- **Latency stays far under 30 s** for every strategy, so prompt choice is a pure accuracy decision here.

This motivates §4 (adaptive routing) and the agentic-tools study in §5.


---
# §4 · Adaptive prompt routing — a small-LLM reasoning study  *(the depth piece)*

The deeper question behind §3: for a small model, does routing each question to a prompt chosen for its
*reasoning shape* beat forcing one universal prompt — and **when does explicit reasoning help vs hurt?**
Four conditions (universal / generic-CoT / structured / adaptive) over a labelled 8-category reasoning
set, with a category×condition heatmap, a latency-vs-accuracy view, a routing-accuracy check, and a
failure taxonomy.

# 04 · Adaptive prompt routing — a small-LLM reasoning study

**Research question:** for a small model (Qwen2.5-7B-Instruct), does routing each question to a prompt
chosen for its *reasoning shape* beat forcing one universal prompt on everything? And the deeper one:
**when does explicit reasoning help, and when does it hurt?**

The motivating failure is real and on the leaderboard: the Maths run died at level 9 on a clock-chime
*interval-counting* question — `cot_v2`'s hard "≤3 steps" brevity cap made the model **guess before it
had counted**. That same cap is what keeps verbose stats questions from timing out. One prompt cannot
serve both regimes — which is exactly the hypothesis this experiment tests.

We isolate ONE variable — the **prompt strategy** — and compare four conditions over a labelled
8-category reasoning set:

| condition | prompt |
|---|---|
| **A_universal** | one universal prompt (production `few_shot_v1`) |
| **B_generic_cot** | always plain "think step by step" |
| **C_structured** | always structured enumeration |
| **D_adaptive** | `ReasoningRouter` picks per question |

No retrieval, no calculator here — *only* the prompt changes between conditions, so the effect we
measure is the prompt's alone.

## 2 · Load the labelled reasoning set
40 MCQs across 8 categories, each with a hand-verified gold answer **and** a gold reasoning-category label (the truth the router is scored against). The clock-chime question that broke the live Maths run is `ic-001`.

In [ ]:
from experiments.adaptive_routing import load_reasoning_eval

questions, categories = load_reasoning_eval('data/reasoning_eval.jsonl')
print(f'{len(questions)} questions; categories:')
from collections import Counter
for cat, n in sorted(Counter(categories.values()).items()):
    print(f'  {cat:<22} {n}')
print('\nThe motivating failure, ic-001:')
print(' ', next(q.text for q in questions if q.qid == 'ic-001')[:140], '...')


## 3 · Pick the engine

**Real run (Colab GPU):** `TransformersEngine` loads Qwen2.5-7B 4-bit — the numbers are real.

**Local / no-GPU:** `SimulatedReasoningEngine` — a **deterministic fixture, NOT a model**. Its
correctness comes from a hand-set skill table that *encodes the hypothesis*, so it validates the
**harness** (routing, logging, metrics, figures) end-to-end. It does **not** produce real findings.

In [ ]:
USE_REAL_MODEL = True   # set False to force the simulated fixture

engine = None
if USE_REAL_MODEL:
    try:
        from inference.engine import TransformersEngine
        from config import RunConfig
        cfg = RunConfig.from_yaml('configs/live.yaml')
        engine = TransformersEngine(cfg.model.name, cfg.model.quantization, cfg.model.dtype)
        engine.warmup()
        print('REAL engine loaded:', engine.name)
    except Exception as e:
        print(f'Real engine unavailable ({type(e).__name__}: {e})')
        print('-> falling back to the SIMULATED fixture.')
        engine = None

if engine is None:
    from inference.engine import SimulatedReasoningEngine
    engine = SimulatedReasoningEngine(questions, categories)
    print('\n' + '!' * 72)
    print('!! SIMULATED FIXTURE ENGINE -- validates the HARNESS, not the model.   !!')
    print('!! Real findings need USE_REAL_MODEL=True on a Colab GPU runtime.      !!')
    print('!' * 72)


## 4 · Run the experiment — 4 conditions × 40 questions
Writes one `experiments/adaptive_routing/records.jsonl` (strategy chosen, router verdict, correctness, latency, tokens, reasoning length, raw output).

In [ ]:
from experiments.adaptive_routing import AdaptiveRoutingExperiment

# max_new_tokens=512 (vs the 256 default): the game allows 130s/question, so a longer cap is free and
# fixes the no_answer_parsed truncations where verbose chains never reached the 'Answer:' line.
exp = AdaptiveRoutingExperiment(engine, max_new_tokens=512)
records_path = exp.run(questions, categories, verbose=True)
records_path


### 🔬 Core implementation: adaptive routing (category -> strategy) + a deterministic Maths solver

**(a) Reasoning-shape routing** (`src/classify/reasoning_router.py`) -- first sort the question into one
of 8 reasoning shapes, then pick the prompt from a policy table. The whole thing is `route()` (three
lines) + one policy table:

```python
def route(self, question: Question) -> tuple[ReasoningSignal, str]:
    signal   = self.classifier.classify(question)                 # 8 classes: arithmetic / temporal /
    strategy = self.policy.get(signal.category, self.fallback)    #   interval_counting / discrete_enum /
    return signal, strategy                                       #   factual_qa / commonsense / logical / multi_hop

DEFAULT_ROUTING_POLICY = {
    # recall/commonsense: an explicit chain HURTS (hallucinated justification, drift) -> answer directly
    FACTUAL_QA: "direct_answer",  COMMONSENSE: "direct_answer",
    ARITHMETIC: "generic_cot",                                    # pure computation: take steps, don't over-enumerate
    TEMPORAL_REASONING:   "structured_enumeration_cot",          # list events/cases in order THEN count ->
    INTERVAL_COUNTING:    "structured_enumeration_cot",          #   enumerate first (the clock-chime 0.4->1.0 fix)
    DISCRETE_ENUMERATION: "structured_enumeration_cot",
    LOGICAL_REASONING: "checklist_cot",  MULTI_HOP: "checklist_cot",  # multi-clause/validity: a per-item checklist
}
```

`RoutingPromptBuilder` is a duck-type of `PromptBuilder`: the pipeline still only calls `.build()` and
reads `.strategy`, but the strategy **varies per question** (the EvalRecord logs which prompt actually
ran). Live Maths uses a **conservative policy**: only the counting/temporal shapes with offline evidence
are re-routed; everything else stays on the battle-tested `cot_v2`, keeping regression risk minimal.

**(b) Deterministic Maths solver** (`src/tools/math_solvers.py`) -- runs **before** the LLM; on a hit
(single option, correct by construction) it answers directly and skips the model; otherwise it abstains
(returns None) and the LLM path is untouched -- it can only **add** correct answers, never overwrite the
model:

```python
_SOLVERS = (_solve_finite_field_roots, _solve_ring_characteristic, _solve_gcd,
            _solve_common_divisor_count, _solve_subspace_intersection_dim, _solve_sum_product,
            _solve_reflection_yx, _solve_triangle_sides, _solve_percentage_increase,
            _solve_operator_placement, _solve_weekday, _solve_parallelogram_angle, _solve_arith_expression)

def solve_maths(question: Question):
    if not question.options: return None                 # MCQ only
    q = _norm(question)                                   # normalise unicode minus/dashes first
    for fn in _SOLVERS:                                   # each type-specific solver in turn
        try: res = fn(q)
        except Exception: res = None                      # crash-safe: an exception counts as abstain
        if res is not None: return res                    # first deterministic hit wins
    return None                                           # all abstain -> hand off to the LLM
```

The cells below are the full offline experiment: 4 conditions x 40 questions, with a
category x condition heatmap, the latency-accuracy trade-off, routing-label accuracy, and a failure
taxonomy.


## 5 · Headline — does adaptive (D) beat the rest, and at what cost?

In [ ]:
import pandas as pd
from experiments import analysis
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 20)

df = analysis.load_records(records_path)
comp = analysis.strategy_comparison_table(df)
comp.round(3)


In [ ]:
from IPython.display import Image, display
analysis.plot_condition_accuracy_bars(df)
display(Image('experiments/adaptive_routing/fig_condition_accuracy.png'))


## 6 · WHERE each prompt helps or hurts — category × condition heatmap
The whole hypothesis in one figure: a prompt that is green on one row can be red on another.

In [ ]:
analysis.plot_category_heatmap(df)
display(Image('experiments/adaptive_routing/fig_category_heatmap.png'))
analysis.category_accuracy_matrix(df).round(2)


## 7 · The cost of reasoning — latency vs accuracy

In [ ]:
analysis.plot_latency_accuracy(df)
display(Image('experiments/adaptive_routing/fig_latency_accuracy.png'))


## 8 · The oracle the router chases — best fixed strategy per category
If the per-category best fixed strategy and the adaptive arm agree, the router is doing its job.

In [ ]:
analysis.best_strategy_per_category(df)


## 9 · Did the classifier label the reasoning shape correctly?
Routing accuracy + a confusion table (true category vs the category the classifier inferred).

In [ ]:
rep = analysis.routing_report(df)
print('overall routing accuracy:', round(rep['overall_routing_accuracy'], 3))
rep['confusion']


## 10 · Failure taxonomy — *why* each prompt got things wrong
Heuristic labels from the raw output: overthinking (recall Q over-reasoned), boundary_error (counting off-by-one), skipped_case (never enumerated), arithmetic_drift, no_answer_parsed, hallucinated/other.

In [ ]:
analysis.failure_taxonomy(df)


## 11 · Findings & recommendation

**Headline:**
- **Adaptive routing (D) should beat every single fixed prompt (A/B/C)** — because no one prompt is
  best across all categories, and D picks the per-category winner.
- **Structured enumeration is the cure for interval-counting / temporal questions** (the clock-chime
  family) but **hurts factual recall** (overthinking) — never make it the universal default.
- **Direct answering wins on factual_qa / commonsense**; **checklists win on logic / multi-hop**.
- The router's weakest spot is **multi_hop** (it surface-overlaps with factual recall).

**Recommendation for the live game:** for the live **Maths** pipeline (§8), use a
per-question router that sends interval-counting / temporal / enumeration questions to
`structured_enumeration_cot` and leaves concept/stats questions on the current chain — exactly the
regime split `cot_v2`'s single brevity cap conflated.

## 12 · Focused A/B — the level-11 induction death (logic questions)

The live Maths run got past the counting deaths (structured routing worked) and died at **level 11** on
a strong-induction question (`log-ind-001`, the real qid 6737): `cot_v2` wrote a 24-token wrong one-liner
("n0 is a counterexample, so all higher values are also counterexamples") — its **brevity cap gagged the
reasoning**, the same root cause as the clock-chime death.

This section pits four prompts over the **logic questions** (the original `lr-*`, five induction /
contrapositive `log-ind-*`, and three "which-of-the-following-is-true" stats look-alikes `log-stat-*`):

| arm | prompt |
|---|---|
| `cot_v2` | the current Maths fallback (≤3-step brevity cap) |
| `generic_cot` | step-by-step, **no** cap |
| `checklist_cot` | per-option verification checklist |
| `checklist_sc5` | checklist + **self-consistency** vote (n=5) — affordable under the 130s budget |

**Decision rule:** adopt the arm that fixes the induction (`log-ind-*`) questions **without** regressing
the stats (`log-stat-*`) ones (those currently win on `cot_v2`).

In [ ]:
from experiments.adaptive_routing import AdaptiveRoutingExperiment, LOGIC_FOCUS_CONDITIONS

# All logic questions (lr-* + log-ind-* + log-stat-*). Same engine as above (real Qwen on Colab).
logic_qs = [q for q in questions if categories[q.qid] == 'logical_reasoning']
print(f'{len(logic_qs)} logic questions:', [q.qid for q in logic_qs])

logic_exp = AdaptiveRoutingExperiment(
    engine, conditions=LOGIC_FOCUS_CONDITIONS,
    out_dir='experiments/adaptive_routing_logic', max_new_tokens=512,
)
logic_path = logic_exp.run(logic_qs, categories, verbose=True)
logic_path

### 12.1 · Per-condition accuracy on logic questions

In [ ]:
import pandas as pd
ldf = analysis.load_records(logic_path)
order = [c.name for c in LOGIC_FOCUS_CONDITIONS]
ldf.groupby('condition')['correct'].agg(accuracy='mean', n='count').reindex(order)

### 12.2 · Per-question correctness (the decision table)
`1.0` = correct. Look at the `log-ind-*` rows (does a prompt fix the induction death?) vs the `log-stat-*` rows (does it regress the stats look-alikes?).

In [ ]:
piv = ldf.pivot_table(index='qid', columns='condition', values='correct').reindex(columns=order)
# order rows: induction first, then stats, then the original lr-*
row_order = [q.qid for q in logic_qs]
piv = piv.reindex(row_order)
piv

### 12.3 · Read-out & decision

- If `checklist_cot` (or `checklist_sc5`) flips `log-ind-001` (qid 6737) and the other `log-ind-*` to
  correct **while keeping every `log-stat-*` correct**, then change the live Maths policy so
  `logical_reasoning -> checklist_cot` (edit `MATHS_LIVE_POLICY` in `src/classify/reasoning_router.py`).
- If checklist **regresses** the stats questions, keep `cot_v2` for them and consider self-consistency
  on `cot_v2` instead, or a finer classifier split (induction vs stats-conclusion).
- The classifier already routes induction questions to `logical_reasoning` (fixed this turn — they used
  to leak to `arithmetic` via the incidental `+1`), so whatever wins here can actually be targeted live.

## 13 · Probe — does explicit logical-DIRECTION prompting break the induction ceiling?

§5c showed the level-11 death (qid 6737) survives `cot_v2`, `generic_cot`, `checklist_cot` AND a
5-vote self-consistency: a **systematic directional misconception** (the model thinks falsity propagates
*forward*). This probe adds **`implication_cot`** — the one prompt that scaffolds the direction: write
`P -> Q`, write the valid contrapositive `not Q -> not P`, and explicitly forbid the converse `Q -> P`
and inverse `not P -> not Q`.

**Go/no-go:** if `implication_cot` flips `log-ind-001` (= 6737), `lr-001`, or `log-ind-003` that the
others miss, the directional hypothesis is worth the full build (trap-labelled dataset + failure
logging). If it moves nothing, 6737 is a capability ceiling and we stop.

In [ ]:
from experiments.adaptive_routing import AdaptiveRoutingExperiment, IMPLICATION_PROBE_CONDITIONS

logic_qs = [q for q in questions if categories[q.qid] == 'logical_reasoning']
probe_path = AdaptiveRoutingExperiment(
    engine, conditions=IMPLICATION_PROBE_CONDITIONS,
    out_dir='experiments/adaptive_routing_implication', max_new_tokens=512,
).run(logic_qs, categories, verbose=True)
probe_path

### 13.1 · Per-condition accuracy + per-question decision table

In [ ]:
import pandas as pd
pdf = analysis.load_records(probe_path)
order = [c.name for c in IMPLICATION_PROBE_CONDITIONS]
display(pdf.groupby('condition')['correct'].agg(accuracy='mean', n='count').reindex(order))
piv = pdf.pivot_table(index='qid', columns='condition', values='correct').reindex(columns=order)
piv = piv.reindex([q.qid for q in logic_qs])
piv

### 13.2 · The decisive rows — did the scaffold fix what the others missed?

In [ ]:
# Focus on the questions the other prompts FAIL: 6737 (log-ind-001), lr-001, log-ind-003.
focus = ['log-ind-001', 'lr-001', 'log-ind-003']
print(piv.reindex(focus).to_string())
print()
for qid in focus:
    row = pdf[(pdf.qid==qid) & (pdf.condition=='implication_cot')]
    if len(row):
        r = row.iloc[0]
        print(f'=== {qid}  implication_cot -> {r.predicted_answer} (gold {r.gold_answer})  '
              f'{"CORRECT" if r.correct else "WRONG"} ===')
        print(r.raw_output[:700]); print()

### 13.3 · Verdict

- **implication_cot fixes ≥1 of the focus rows** (esp. 6737) → proceed to the full build: a trap-labelled
  directional-logic dataset (converse / inverse / contrapositive / induction-up / induction-down), the
  metadata-driven failure logging (converse/inverse/contrapositive/induction-direction confusion), and
  wiring `logical_reasoning -> implication_cot` into the Maths policy.
- **implication_cot moves nothing** → 6737 is a 7B capability ceiling; record the negative result and stop.

---
# §5 · Tools + ensemble voting  *(Phase 3 & 5 — agentic AI)*

The assignment encourages agentic techniques: tool calls + multi-model / self-consistency voting. We
**implemented and evaluated both**, and the honest result is that **neither is active in our final live
configuration** — which is itself a finding:

- **Self-consistency** (vote over N sampled CoT chains): **dropped everywhere** — every live pipeline runs
  `self_consistency_n = 1`. On Maths, `cot_v2` + SC(n=3) timed out at ~41 s on a question it had answered
  *correctly*, and the three chains shared the same slip so voting could not self-correct. Single-pass
  `cot_v2` is strictly better here (see §4 / §8).
- **Calculator** (safe-AST tool): **wired into the non-Maths pipelines but effectively never fires.** It is
  gated by `needs_calculator`, which only triggers on arithmetic-shaped questions — and those occur only in
  the Maths race, whose pipeline sets `tools=None` (at n=1 the tool clobbered correct `cot_v2` chains on
  stats questions). On the knowledge / News races it is a harmless no-op.

So for a 7B model under a 30 s wall, **a single well-engineered `cot_v2` pass beat both** — that is the
lesson. The two components below are shown because we built and studied them; the code stays in the repo.


### 🔬 Core implementation: multi-model / self-consistency voting + a safe calculator tool (agentic AI)

The assignment encourages **agentic** techniques: tool calls + multi-model combination. Our two core
components, source shown directly below.

**(a) Majority vote** (`src/agent/voting.py`) -- one primitive serves two callers: (1) self-consistency
(N sampled CoT chains from the same model vote) and (2) the Phase-5 ensemble (different models, one vote
each). Confidence is the **vote share**, a genuine calibration signal (2/3 votes = the uncertainty of
"two agree, one dissents"):

```python
def majority_vote(predictions: list[Prediction]) -> Prediction:
    if not predictions:
        raise ValueError("majority_vote of an empty list ...")
    counts = Counter(p.answer for p in predictions)          # votes per answer
    top = max(counts.values())
    tied = [a for a, c in counts.items() if c == top]
    if len(tied) > 1:                                        # tie -> break by each side's mean confidence
        winner_answer = max(tied, key=lambda a: _mean_conf(a))
    else:
        winner_answer = tied[0]
    winners = [p for p in predictions if p.answer == winner_answer]
    rep = max(winners, key=lambda p: p.confidence)           # the most confident winning sample represents
    vote_share = top / len(predictions)                      # confidence = vote share (a real calibration signal)
    return Prediction(qid=rep.qid, answer=winner_answer, confidence=vote_share,
                      tool_used=next((p.tool_used for p in winners if p.tool_used), None), ...)
```

**(b) Safe calculator tool** (`src/tools/calculator.py`) -- we **never** give the model the dangerous
power of `eval()`; we only walk an arithmetic AST, safe by construction:

```python
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.FloorDiv: operator.floordiv, ast.USub: operator.neg, ast.UAdd: operator.pos}

def _eval(node):                       # only numbers and arithmetic allowed, everything else rejected
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("Forbidden, this expression is.")

def calculate(expression: str) -> float:           # "12 * (3 + 4) / 2" -> 42.0
    return _eval(ast.parse(expression, mode="eval").body)
```

In the pipeline the calculator is a **verifier** (see `_run_calculator_tool`): the model emits a JSON
call, we compute the value, and we override the original CoT answer **only if** it uniquely matches one
option (+/-0.5%) -- so "an answer that happens to be a conclusion using numbers" (like a t-test) is not
wrongly rewritten.

> **Final-config status (honest).** Despite the implementations above, **neither component is active in our
> submitted live run**: `self_consistency_n = 1` in every pipeline (SC dropped for latency), and the
> calculator is gated off on the only race it would apply to (Maths uses `tools=None`). We keep both as
> implemented-and-evaluated study artifacts — the takeaway is that single-pass `cot_v2` outperformed both.


---
# §6 · RAG — live web retrieval, raw content only  *(Phase 4)*

News (and the knowledge races) sit behind a routed retriever. **Rule-compliant**: only free sources that
return **raw, non-generated** content — Google News RSS + the Guardian Open Platform API + headless-Chromium
article bodies, with a query cascade, date re-ranking, and per-doc relevance filtering. Below: the core
implementation, a live stress-test (N rounds, each logged), and a **RAG-vs-no-RAG ablation** on the
retrieval races.

> The `TARGET` switch picks the race; outputs preserved here are from our stress-test run. Set
> `TARGET='news'` and re-run §6 to reproduce the News numbers.

### 🔬 Core implementation: live News RAG -- a keyword retrieval cascade + date re-ranking + body focusing

News (competition 5) uses live web RAG. **Strictly within the assignment rules**: only free sources that
return **raw, non-generated** content (Google News RSS, the Guardian API's `bodyText`, and
headless-Chromium-scraped article body HTML) -- never any LLM-generated answer. The source names are
credited in the video.

The core is the **retrieval cascade** in `WebSearchRetriever.retrieve()` (`src/retrieval/retriever.py`):
a full-sentence query is often AND-matched by gnews into 0 hits, so it falls back step by step to
shorter, sharper queries, re-ranks by the question date, and finally fetches the body:

```python
def retrieve(self, question: Question) -> list[RetrievedDoc]:
    query = _query_from_question(question)

    # 1) primary query + date window (after:..before:.., drops time-irrelevant noise); too-narrow 0 hits
    #    -> retry once without the window
    window = _gnews_date_window(question)
    items = self._gnews_items(query + window) or self._gnews_items(query)

    # 2) keyword re-search: full sentence missed / off-topic -> use a short entity/noun query (>=2 words,
    #    so a single generic word doesn't drag back a pile of off-topic news)
    if not items or not _headlines_on_topic(items, question):
        kwq = _keyword_query(question)
        if kwq and len(kwq.split()) >= 2:
            kw_items = self._gnews_items(kwq + window) or self._gnews_items(kwq)
            if kw_items:
                items = kw_items + [it for it in items if it not in kw_items]

    # 3) option-augmented re-search: still off-topic -> splice option keywords ("+missiles"/"+CEPI") into
    #    the query, to surface the answer article a bare query missed
    if not _headlines_on_topic(items, question):
        opt_terms = _option_query_terms(question)
        if opt_terms:
            aug = (_strip_body_details(query) + " " + opt_terms).strip()
            aug_items = self._gnews_items(aug + window) or self._gnews_items(aug)
            if aug_items: items = aug_items + [...]

    # 4) re-rank by how close each pubDate is to the question date -- a stale hit (e.g. a 2019 article)
    #    must never outrank the current one
    items = _rerank_by_recency(items, question)
    headlines = [RetrievedDoc(text=t[:self.char_limit], source="google_news_rss") for t,_,_ in items[:top_k]]

    # 5) fetch the body (exact numbers / "who said it" live in the body): Guardian API first (one ~0.2s
    #    call for raw bodyText), else browser/ddg fallback. Each doc is relevance-filtered, blocking
    #    captcha/Cloudflare block pages and off-topic bodies
    bodies = []
    if self.fetch_bodies > 0 and self.body_mode != "off":
        keywords = _question_keywords(question)
        guardian = [d for d in self._guardian_bodies(query, question, self.fetch_bodies)
                    if _relevance(d.text, keywords) >= _GUARDIAN_RELEVANCE_MIN]
        if guardian:
            bodies = guardian
        else:                              # non-Guardian stories -> wide web path, also relevance-filtered
            fetched = self._fetch_bodies_via_browser([lnk for _,lnk,_ in items], ...) or ...
            bodies = [d for d in fetched if _relevance(d.text, keywords) >= _GUARDIAN_RELEVANCE_MIN] or fetched

    docs = bodies + headlines                # bodies first (richer), headlines after
    return docs or self._ddg_search(query)   # all empty (gnews down) -> DDG fallback, else route to Wikipedia
```

The fetched body is then trimmed by `_focus_body()` around the question keywords to a budget length
(keeping the answer sentence inside the window), then handed to the prompt builder for grounding.
The **RAG vs no-RAG ablation** later in this section confirms retrieval is a net gain (the model's prior
is often wrong).


In [ ]:
from config import RunConfig

config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# --- Pick which race to stress-test -------------------------------------------------------------
TARGET = 'science'   # any key of COMP_IDS below (e.g. 'science', 'history', 'philosophy', 'news', 'math', 'entertainment')
# ------------------------------------------------------------------------------------------------

# Every competition the harness can drive. The value is the server competition_id (confirm in the
# post-login list printed by cell 10). Order per configs/live.yaml: 0 Entertainment . 1 History .
# 2 Science . 3 Maths . 4 Philosophy . 5 News.
COMP_IDS = {
    'entertainment': 0,   # Entertainment                 -- knowledge RAG (FAISS/Wikipedia), few_shot_entertainment.
    'history':       1,   # Ancient History and Politics  -- knowledge RAG; in _HIGH_RETRIEVAL_TOPICS (always retrieves).
    'science':       2,   # Science and Nature            -- knowledge RAG; in _HIGH_RETRIEVAL_TOPICS (added 2026-06-02).
    'math':          3,   # Maths                         -- adaptive cot routing, NO RAG / NO tools.
    'philosophy':    4,   # Philosophy and Psychology     -- knowledge RAG; on-demand gate (not forced).
    'news':          5,   # News                          -- live-web RAG (Guardian API + headless Chromium).
}
assert TARGET in COMP_IDS, f'unknown TARGET {TARGET!r}; pick one of {sorted(COMP_IDS)}'

# Knowledge races -- the same `routed` retriever sends them to FAISS/Wikipedia (topic != News). They share
# one recipe; only Entertainment swaps in its own pop-culture exemplars (few_shot_entertainment).
KNOWLEDGE_TARGETS = {'entertainment', 'history', 'science', 'philosophy'}
# Anything that uses a retriever at all (knowledge RAG OR News web RAG). Only Maths runs retrieval-free, so
# the retrieval-flavoured diagnostics / ablation below key off this; Maths takes the reasoning-chain path.
RETRIEVAL_TARGET = TARGET != 'science'

COMP_ID    = COMP_IDS[TARGET]
NUM_ROUNDS = 10            # how many live games to play, back to back.
PAUSE_S    = 8.0          # polite gap between games (PDF: no rapid consecutive requests).

config.game.competition_id = COMP_ID
config.game.game_mode = 'text'

# The Guardian Open Platform key -- from a Colab secret (NEVER hardcoded). Only News uses it, but harmless
# to set always. With it, the Guardian fast body path is armed; the relevance gate keeps it ONLY on-topic.
try:
    from google.colab import userdata as _ud
    config.retrieval.guardian_api_key = _ud.get('guardian_key') or ''
except Exception:
    config.retrieval.guardian_api_key = config.retrieval.guardian_api_key or ''

print('TARGET:', TARGET, '| competition_id:', COMP_ID)
print('rounds:', NUM_ROUNDS, '| pause between:', PAUSE_S, 's')
print('aim_seconds:', config.game.aim_seconds, '| model:', config.model.name, '|', config.model.quantization)
if TARGET == 'news':
    print('News pipeline: few_shot_v1 + live-web RAG.')
    print('RAG:', 'ON' if config.retrieval.enabled else 'OFF', '| source:', config.retrieval.source,
          '| news_body_mode:', config.retrieval.news_body_mode, '| fetch_bodies:', config.retrieval.news_fetch_bodies)
    print('Guardian API:', 'KEY SET' if config.retrieval.guardian_api_key else 'no key -> News uses browser')
elif TARGET in KNOWLEDGE_TARGETS:
    strat = 'few_shot_entertainment' if TARGET == 'entertainment' else config.prompt_strategy
    print(f'{TARGET.capitalize()} pipeline: {strat} + RAG (routed -> FAISS/Wikipedia, gated by needs_retrieval).')
    print('RAG:', 'ON' if config.retrieval.enabled else 'OFF', '| source:', config.retrieval.source,
          '| top_k:', config.retrieval.top_k, '| min_score:', config.retrieval.min_score,
          '| bm25:', config.retrieval.bm25_index_path or 'off')
else:  # math
    print('Maths pipeline: adaptive cot_v2 / structured_enumeration_cot routing, NO retrieval, NO calculator,'
          ' max_new_tokens=450 (post-B3, 30s-wall safe).')


## 3 · Load + warm up the model

In [ ]:
import time, gc
from inference.engine import TransformersEngine
import torch

# --- Free any leftover model from a PREVIOUS run BEFORE (re)loading ---------------------------------
# The Maths "model comparison" cell (TARGET='math') loads a SECOND model `engine_math` and frees `engine`.
# If that model's VRAM is still resident, the 4-bit base model no longer fits and bitsandbytes spills
# modules to the CPU -> the "Some modules are dispatched on the CPU or the disk" ValueError you hit.
# So drop `engine_math` and clear the CUDA cache first. A HEALTHY `engine` we keep (no reload, no wait).
if 'engine_math' in globals():
    del engine_math
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

t0 = time.perf_counter()
if 'engine' not in globals():
    engine = TransformersEngine(model_name=config.model.name,
                                quantization=config.model.quantization,
                                dtype=config.model.dtype)
    print(f'Model loaded in {time.perf_counter() - t0:.1f}s')
else:
    print('engine already in VRAM, skipping load.')

t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')


## 4 · Wire the pipeline for `TARGET` + log in to the game

In [ ]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder, RoutingPromptBuilder
from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
from agent.pipeline import QAPipeline
from tools import default_tools, solve_maths
from retrieval import build_retriever

if RETRIEVAL_TARGET:
    # Phase 4 RAG: the routing retriever. News (post-cutoff) `routed` sends questions to the live web
    # (Guardian API + relevance gate -> headless-Chromium on the gnews link); every KNOWLEDGE race
    # (Entertainment / History / Science / Philosophy) routes the SAME `routed` retriever to FAISS/Wikipedia
    # (topic != News). `needs_retrieval` gates it per question.
    retriever = build_retriever(config.retrieval)
    print('RAG:', (f'ON  source={config.retrieval.source}  top_k={config.retrieval.top_k}') if retriever else 'OFF')

    # Per-race strategy: Entertainment gets its OWN `few_shot_entertainment` (pop-culture exemplars +
    # domain instruction, mirrors the §8 pipeline_entertainment); every other knowledge race AND News
    # keep config.prompt_strategy (few_shot_v1). All: RAG + the classifier-gated calculator no-op.
    strategy = 'few_shot_entertainment' if TARGET == 'entertainment' else config.prompt_strategy
    pipeline = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=strategy),
        classifier=QuestionClassifier(),
        retriever=retriever,
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )
    print(f'{TARGET.capitalize()} pipeline wired:', strategy, '+ RAG (routed) + classifier-gated tools')

else:  # math -- EXACTLY the `pipeline_maths` recipe in §8 (comp 3).
    # Adaptive routing: counting / temporal / discrete-enumeration -> structured_enumeration_cot; everything
    # else (arithmetic, logic, concept/stats) -> the cot_v2 fallback. NO retrieval (it only distracts on
    # Maths), NO calculator (at n=1 it clobbers the chain on numeric Qs), single-pass. max_new_tokens=450
    # (raised from 300 post-B3: only short time-interval Qs reach structured enum now, so chains finish in
    # 5-10s -- big headroom; 450 tok ~= 28s worst-case, dial to 400 if any Maths turn nears the 30s wall).
    pipeline = QAPipeline(
        engine=engine,
        prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
        classifier=QuestionClassifier(),
        retriever=None,
        tools=None,
        solver=solve_maths,   # deterministic type-specific solver: short-circuits the LLM on solvable types.
        latency_budget_s=config.latency_budget_s,
        max_new_tokens=450,
    )
    print('Maths pipeline wired: ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2)'
          ' + 450 tokens + single-pass + NO retrieval + NO general-calculator + DETERMINISTIC solver (solve_maths)')

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')
PASSWORD = userdata.get('password')

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions + ids (safe -- starts no timer). Confirm the target's id here (News=5, Maths=3).
for c in game_client.list_competitions():
    print('  id=', c.id, '|', c.name, '| max_levels=', getattr(c, 'max_levels', '?'))

## 5 · ▶ Play 10 live rounds  (each its own logged run)

Plays the `TARGET` game `NUM_ROUNDS` times. Each round writes its own run dir `{target}_r{NN}` (so no
round overwrites another — the LiveRunner truncates *within* a run_id). One round failing (a rate-limit, a
network blip) is caught and logged as a gap — the loop carries on.

In [ ]:
import time
from evaluation.runner import run_session

LOG_ROOT = os.path.join(REPO_ROOT, 'experiments', 'runs')
round_runs = []   # [(round_no, run_path-or-None)]

for r in range(1, NUM_ROUNDS + 1):
    config.run_id = f'{TARGET}_r{r:02d}'
    print(f'\n===== ▶ ROUND {r}/{NUM_ROUNDS}  (run_id={config.run_id}) =====')
    try:
        path = run_session(pipeline, config, game_client=game_client, log_root=LOG_ROOT)
        round_runs.append((r, path))
        print('   round log:', path)
    except Exception as e:
        round_runs.append((r, None))
        print(f'   ⚠️ round {r} FAILED ({type(e).__name__}: {e}) -- logged as a gap, the loop continues.')
    if r < NUM_ROUNDS:
        time.sleep(PAUSE_S)   # polite gap between live games.

print('\nAll rounds done. Logged runs:', [p for _r, p in round_runs if p])

## 6 · Analysis — per-round scores + every wrong question (with diagnostics)

Aggregates all rounds and **saves** the consolidated records to `experiments/{TARGET}_test/` so they ride
back to the repo. Three artifacts: a per-round summary, every question, and the wrong questions alone.
- **News**: wrong questions carry the retrieved evidence text (retrieval-miss vs grounding-miss diagnosis)
  and a retrieval source mix.
- **Math**: wrong questions carry the reasoning chain (`raw_output`), which routing strategy fired
  (`prompt_strategy`), and the output token count — so you can see truncation at the 300-token cap.

In [ ]:
import json, collections
from pathlib import Path
import pandas as pd

OUT_DIR = Path(REPO_ROOT) / 'experiments' / f'{TARGET}_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _read_round(path):
    """One round's records.jsonl -> list of dict rows ([] if missing/empty)."""
    if not path:
        return []
    p = Path(path) / 'records.jsonl'
    if not p.exists():
        return []
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]

def _sources(row):
    """The retrieval SOURCE mix for a row, from retrieved_snippets ('[theguardian.com#..] ' prefix)."""
    out = collections.Counter()
    for s in (row.get('retrieved_snippets') or []):
        if isinstance(s, str) and s.startswith('['):
            out[s[1:].split('#', 1)[0].split(']', 1)[0]] += 1
    return dict(out)

# --- Gather every round ---
summary_rows, all_q, wrong_q = [], [], []
for r, path in round_runs:
    rows = _read_round(path)
    graded = [x for x in rows if x.get('correct') is not None]
    n_correct = sum(1 for x in graded if x.get('correct') is True)
    reached = [x.get('reached_level') for x in rows if x.get('reached_level') is not None]
    summary_rows.append({
        'round': r,
        'answered': len(rows),
        'correct': n_correct,
        'graded': len(graded),
        'accuracy': (n_correct / len(graded)) if graded else float('nan'),
        'reached_level': max(reached) if reached else None,
    })
    for x in rows:
        rec = {'round': r, **x, 'sources': _sources(x)}
        all_q.append(rec)
        if x.get('correct') is False:
            wrong_q.append(rec)

# --- Per-round summary + overall ---
summary = pd.DataFrame(summary_rows)
print(f'PER-ROUND SUMMARY ({TARGET})')
print(summary.to_string(index=False))
tot_c = int(summary['correct'].sum()); tot_g = int(summary['graded'].sum())
lv = [s['reached_level'] for s in summary_rows if s['reached_level'] is not None]
print(f"\nOVERALL: {tot_c}/{tot_g} graded = {tot_c / tot_g:.1%}" if tot_g else '\nOVERALL: no graded answers')
if lv:
    print(f"reached_level over {len(lv)} rounds: min={min(lv)} max={max(lv)} mean={sum(lv)/len(lv):.1f} | {sorted(lv, reverse=True)}")

if RETRIEVAL_TARGET:
    # --- Retrieval source mix across ALL questions (did the gate route to the browser? did docs land?) ---
    src_total = collections.Counter()
    for x in all_q:
        src_total.update(x['sources'])
    print('\nRETRIEVAL SOURCE MIX (doc count across all', len(all_q), 'questions):', dict(src_total))
    fired = sum(1 for x in all_q if x.get('retrieval_used'))
    print(f"retrieval fired on {fired}/{len(all_q)} questions")
else:  # math -- which routing strategy fired, and how close to the 300-token cap the chains ran.
    strat_mix = collections.Counter(x.get('prompt_strategy') or '?' for x in all_q)
    print('\nROUTING STRATEGY MIX (across all', len(all_q), 'questions):', dict(strat_mix))
    toks = [x.get('tokens_out', 0) for x in all_q if x.get('tokens_out')]
    if toks:
        capped = sum(1 for t in toks if t >= 300)
        print(f"tokens_out: min={min(toks)} max={max(toks)} mean={sum(toks)/len(toks):.0f}"
              f" | hit the 300 cap on {capped}/{len(toks)} questions (likely truncated before 'Answer:')")

In [ ]:
# --- Every WRONG question, with the right diagnostics for the target ---
#   News / Entertainment: the retrieved EVIDENCE (retrieval-miss vs grounding-miss check).
#   Math: the reasoning chain (raw_output), the routing strategy that fired, and tokens_out (cap = 300).
print(f"{'=' * 78}\nEVERY WRONG QUESTION  ({len(wrong_q)} across {NUM_ROUNDS} rounds)\n{'=' * 78}")
for x in wrong_q:
    opts = x.get('options') or {}
    pick = x.get('predicted_answer')
    if RETRIEVAL_TARGET:
        print(f"\n[round {x['round']}] qid={x['qid']} reached_level={x.get('reached_level')} "
              f"lat={x.get('latency_s', 0):.1f}s sources={x['sources']}")
    else:
        toks = x.get('tokens_out', 0)
        print(f"\n[round {x['round']}] qid={x['qid']} level={x.get('level')} reached_level={x.get('reached_level')} "
              f"lat={x.get('latency_s', 0):.1f}s strategy={x.get('prompt_strategy')} "
              f"tokens_out={toks}{'  <-- HIT 300 CAP (truncated?)' if toks and toks >= 300 else ''}")
    print(f"Q: {x['question_text']}")
    for k, v in opts.items():
        print(f"   {k}. {v}" + ('  <-- our pick (WRONG)' if k == pick else ''))
    if RETRIEVAL_TARGET:
        snips = x.get('retrieved_snippets') or []
        if snips:
            print('   -- retrieved evidence --')
            for s in snips:
                print(f"      {str(s)[:500]}")
        else:
            print('   (no retrieved evidence logged)')
    else:  # math -- the model's reasoning chain (truncation / wrong set-up shows here).
        raw = (x.get('raw_output') or '').strip()
        print('   -- reasoning chain --')
        print('      ' + (raw[:900].replace('\n', '\n      ') if raw else '(empty)'))

# --- SAVE the consolidated artifacts (ride back to the repo via experiments/) ---
summary.to_csv(OUT_DIR / 'summary.csv', index=False)
with open(OUT_DIR / 'all_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in all_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
with open(OUT_DIR / 'wrong_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in wrong_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
print(f"\nSaved -> {OUT_DIR}/  (summary.csv, all_questions.jsonl [{len(all_q)}], wrong_questions.jsonl [{len(wrong_q)}])")

## 8 · RAG vs no-RAG ablation — retrieval targets only — do questions answer better WITHOUT retrieval?

**Runs only for retrieval targets (`TARGET` in `entertainment` / `news`)** — Maths uses no retrieval, so there is nothing to ablate, and this cell is skipped there.

Some News questions are really **knowledge/historical** ("which US president visited China in 2008..") —
the model may know the answer from its **own training**, and an off-topic retrieved article can *mislead*
it (grounding on junk). This re-answers the SAME questions **with** retrieval and **without**, side by
side, so we can see where RAG helps vs hurts. Annotate `KNOWN_GOLD` for questions you can verify by hand
to get a score (live games hide the gold).

In [ ]:
if not RETRIEVAL_TARGET:
    print(f"RAG-vs-noRAG ablation needs a retriever (Maths uses none) -- skipping for TARGET = {TARGET}.")
else:
    import json
    from schemas import Question, QuestionType
    from agent.pipeline import QAPipeline
    from prompting.builder import PromptBuilder
    from classify.classifier import QuestionClassifier
    from tools import default_tools

    # Twin of THIS target's RAG pipeline, but retriever=None -> pure parametric knowledge (no RAG).
    _strategy = 'few_shot_entertainment' if TARGET == 'entertainment' else config.prompt_strategy
    pipeline_norag = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=_strategy),
        classifier=QuestionClassifier(),
        retriever=None,                 # <- the only difference from the RAG `pipeline`.
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )

    # Questions to probe. Default: the wrong questions THIS News run logged. Point SRC elsewhere to test others.
    SRC = os.path.join(REPO_ROOT, 'experiments', f'{TARGET}_test', 'wrong_questions.jsonl')
    rows = [json.loads(l) for l in open(SRC, encoding='utf-8') if l.strip()]

    # Known gold BY TEXT substring (resolved to the option letter at runtime -> robust to option shuffling).
    # Fill in the ones you can verify; un-annotated rows are still printed for eyeballing.
    # NOTE: the entries below are News examples -- for Entertainment, replace them with your own verified golds.
    KNOWN_GOLD = {
        '10851': 'George W. Bush',     # Bush attended a Beijing church service, 2008 Olympics
        '10659': 'CEPI',               # Coalition for Epidemic Preparedness Innovations
        '12017': 'Cannes',             # Cannes Film Festival opens ~May 12
        '10645': '161',                # Pentagon released ~16x declassified UFO files
        '10747': '1.5 million',        # Labour's housing pledge
    }

    def _build_q(r):
        try: qt = QuestionType(r.get('qtype', 'mcq'))
        except Exception: qt = QuestionType.MCQ
        return Question(qid=r['qid'], text=r['question_text'], options=r.get('options') or {},
                        qtype=qt, level=r.get('level'), topic=r.get('topic'), language=r.get('language'))

    def _gold_letter(r):
        sub = KNOWN_GOLD.get(str(r['qid']))
        if not sub:
            return None
        for k, v in (r.get('options') or {}).items():
            if sub.lower() in str(v).lower():
                return k
        return None

    print(f"{'qid':>7} | RAG | noRAG | gold | verdict")
    print('-' * 70)
    rag_ok = norag_ok = scored = 0
    for r in rows:
        q = _build_q(r)
        a_rag = pipeline.answer(q).answer          # WITH live retrieval
        a_no  = pipeline_norag.answer(q).answer    # parametric knowledge only
        gold = _gold_letter(r)
        verdict = ''
        if gold:
            scored += 1
            rag_ok   += (a_rag == gold)
            norag_ok += (a_no  == gold)
            verdict = f"RAG {'OK' if a_rag==gold else 'X'} | noRAG {'OK' if a_no==gold else 'X'}"
        print(f"{r['qid']:>7} |  {a_rag}   |  {a_no}    |  {gold or '-'}   | {verdict}")
        print(f"          Q: {r['question_text'][:82]}")

    if scored:
        print(f"\nON {scored} ANNOTATED-GOLD QUESTIONS:   RAG {rag_ok}/{scored}    no-RAG {norag_ok}/{scored}")
    print("\n(Eyeball RAG vs noRAG on the un-annotated rows; add to KNOWN_GOLD to score more.)")

---
# §7 · Model comparison — base 7B vs a math-specialised model  *(model A/B · size vs quality)*

Does a math-specialised open model beat the general base 7B on the Maths race, and at what latency cost?
⚠️ This cell loads a **second** model (frees the first), so run it on its own.

## 10 · Model comparison (Maths-only) — base 7B vs a math-specialised open model

**Runs only when `TARGET == 'math'`.** ⚠️ It loads a SECOND model and FREES `engine` — never run it on an entertainment / news session (it would tear down the live pipeline).

Answers the assignment's *"are certain models better at certain topics?"*. Same Maths questions through both models' **pure-LLM** pipeline (solver OFF — measures the *model*), scored vs hand-verified gold, with flips and >25s flags.

> Open-weight + local only (no API). On a **T4** the two 7B-4bit models can't co-reside, so the base model is freed before the math model loads — **re-run the *Load + warm up the model* cell afterwards** to restore `engine`. ⚠️ This cell must be RUN on Colab (loads a 2nd model, ~5 min); it does nothing if just pulled.

In [ ]:
# Maths-only AND destructive (loads a 2nd model, FREES `engine`). Guard hard: on any non-Maths
# target skip entirely, so an entertainment / news run never tears down its own live engine.
if TARGET != 'math':
    print(f"Model comparison is Maths-only (and frees `engine`) -- skipping for TARGET = {TARGET}.")
else:
    # ============================================================================
    # 9 · MODEL COMPARISON (Maths) -- base Qwen2.5-7B vs a math-specialised open model.
    # Assignment investigation question: "are certain models better at certain topics?"
    # Same Maths questions through BOTH models' PURE-LLM pipeline (solver OFF -> we measure the
    # MODEL, not the deterministic short-circuit), scored vs hand-verified gold, flags flips + >25s.
    # Rules: open-weight, local only (no API). T4 NOTE: two 7B-4bit can't co-reside, so we FREE the
    # base model before loading the math one -> re-run the "Load + warm up the model" cell to restore it.
    # ============================================================================
    import time, json, gc, os
    import torch
    from schemas import Question, QuestionType
    from prompting.builder import RoutingPromptBuilder
    from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
    from classify.classifier import QuestionClassifier
    from agent.pipeline import QAPipeline
    from inference.engine import TransformersEngine

    MATH_MODEL = 'Qwen/Qwen2.5-Math-7B-Instruct'   # the challenger (open-weight only); swap to try others.
    FREE_A_BEFORE_B = True                          # T4: free the base model before loading the math one.

    # --- question set: this run's saved Maths questions if present, else a small embedded fallback ---
    rows, SRC = [], None
    for cand in ('all_questions.jsonl', 'wrong_questions.jsonl'):
        p = os.path.join(REPO_ROOT, 'experiments', 'math_test', cand)
        if os.path.exists(p):
            SRC = p
            rows = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
            break
    if not rows:   # fallback so the cell always runs (knowledge-heavy: where a math model should help)
        SRC = 'embedded fallback'
        rows = [
            {'qid': '6919', 'question_text': 'Statement 1 | Q is an extension field of Z_2. Statement 2 | Every non-constant polynomial over a field has a zero in some extension field.', 'options': {'A': 'True, False', 'B': 'False, False', 'C': 'True, True', 'D': 'False, True'}},
            {'qid': '6713', 'question_text': 'Statement 1 | If H is a subgroup of a group G and a belongs to G, then aH = Ha. Statement 2 | If H is normal of G and a belongs to G, then ah = ha for all h in H.', 'options': {'A': 'True, False', 'B': 'True, True', 'C': 'False, False', 'D': 'False, True'}},
            {'qid': '6910', 'question_text': 'Statement 1 | If T: V -> W is a linear transformation and dim(V) < dim(W) < 1, then T must be injective. Statement 2 | Let dim(V) = n and suppose that T: V -> V is linear. If T is injective, then it is a bijection.', 'options': {'A': 'False, False', 'B': 'False, True', 'C': 'True, True', 'D': 'True, False'}},
            {'qid': '6767', 'question_text': 'Find the order of the factor group (Z_4 x Z_12)/(<2> x <2>)', 'options': {'A': '3', 'B': '4', 'C': '12', 'D': '2'}},
            {'qid': '6886', 'question_text': 'Find all zeros in the indicated finite field of the given polynomial with coefficients in that field. x^5 + 3x^3 + x^2 + 2x in Z_5', 'options': {'A': '0,4', 'B': '0,1', 'C': '0', 'D': '1'}},
        ]
    rows = list({r['qid']: r for r in rows}.values())   # dedupe by qid
    print(f"comparison set: {len(rows)} Maths questions  (source: {SRC})")

    # --- hand-verified gold BY TEXT SUBSTRING (unambiguous within each option set). Add more freely. ---
    KNOWN_GOLD = {
        '6886': '0,4', '6809': '2047', '6962': '5, 14', '6728': '37', '6932': '8', '7021': '0', '6841': '46',
        '6899': 'Friday', '6817': '70', '6767': '4', '6781': '3', '6671': '24', '6688': '15',
        '6919': 'False, True', '6713': 'False, False', '6910': 'True, True', '6912': 'True, True',
        '6894': 'False, False', '6696': 'False, True',
    }

    def _bq(r):
        try: qt = QuestionType(r.get('qtype', 'mcq'))
        except Exception: qt = QuestionType.MCQ
        return Question(qid=r['qid'], text=r['question_text'], options=r.get('options') or {}, qtype=qt,
                        level=r.get('level'), topic=r.get('topic'), language=r.get('language'))

    def _gold(r):
        sub = KNOWN_GOLD.get(str(r['qid']))
        if not sub: return None
        hits = [k for k, v in (r.get('options') or {}).items() if sub.lower() in str(v).lower()]
        return hits[0] if len(hits) == 1 else None

    def _make_pipe(eng):   # PURE-LLM Maths pipeline (solver OFF) -> measures the MODEL itself.
        return QAPipeline(engine=eng, prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
                          classifier=QuestionClassifier(), retriever=None, tools=None, solver=None,
                          latency_budget_s=config.latency_budget_s, max_new_tokens=450)

    def _run(eng, label):
        pipe, out = _make_pipe(eng), {}
        for r in rows:
            t0 = time.perf_counter(); pred = pipe.answer(_bq(r)); out[r['qid']] = (pred.answer, time.perf_counter() - t0)
        print(f"  {label}: {len(out)} answered"); return out

    print(f"\n[A] base model: {config.model.name}")
    resA = _run(engine, 'A')

    if FREE_A_BEFORE_B:
        del engine; gc.collect(); torch.cuda.empty_cache()
        print("   freed base model (re-run the model-load cell to restore `engine` afterwards).")

    print(f"\n[B] loading math model: {MATH_MODEL}")
    engine_math = TransformersEngine(model_name=MATH_MODEL, quantization=config.model.quantization, dtype=config.model.dtype)
    engine_math.warmup()
    resB = _run(engine_math, 'B')

    # --- compare ---
    print(f"\n{'qid':>7} | A | B | gold | note")
    print('-' * 64)
    a_ok = b_ok = scored = Bfix = Bbreak = 0; over = []
    for r in rows:
        q = r['qid']; (aA, latA), (aB, latB) = resA[q], resB[q]; g = _gold(r)
        if latA > 25: over.append((q, 'A', round(latA, 1)))
        if latB > 25: over.append((q, 'B', round(latB, 1)))
        note = ''
        if g:
            scored += 1; oa, ob = (aA == g), (aB == g); a_ok += oa; b_ok += ob
            if ob and not oa: Bfix += 1; note = 'B fixes A'
            elif oa and not ob: Bbreak += 1; note = 'B BREAKS A'
        print(f"{q:>7} | {aA} | {aB} | {g or '-'} | {note}{'  <-DIFF' if aA != aB else ''}")

    if scored:
        print(f"\nON {scored} GOLD-LABELLED:  base-A {a_ok}/{scored} ({a_ok/scored:.0%})   |   math-B {b_ok}/{scored} ({b_ok/scored:.0%})")
        print(f"   B fixes A: {Bfix}   |   B breaks A: {Bbreak}   (net {Bfix-Bbreak:+d})")
    print(f"latency >25s (wall risk): {over or 'none'}")
    print("\nNOTE: this measures the MODEL (solver OFF). Add qids to KNOWN_GOLD to score more.")
    print("To restore the 7B for other cells, RE-RUN the 'Load + warm up the model' cell.")


---
# §8 · Live play — the REAL test  *(the game API + full sweep)*

The actual game, not the dev set. Load the live config, wire the same pipeline, log in with the Colab
secret, then play. The **full sweep** plays one live game in each of the 6 competitions and reports the
reached level + every wrong question with its tool/retrieval/strategy trace. **This is the fresh final
run for the leaderboard.**

## 2 · Load the LIVE run config
`configs/live.yaml` carries `mode: 'live'`. The competition + game mode, here you choose.

Competitions: 0 Entertainment · 1 History · 2 Science · 3 Maths · 4 Philosophy · 5 News.

In [ ]:
from config import RunConfig

# The live config, from YAML we load.
config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# Which competition to play + how, here choose it you do.
config.game.competition_id = 0          # 0..5
config.game.game_mode = 'text'          # 'text' | 'speech'
config.run_id = f'live_comp{config.game.competition_id}'

# The Guardian Open Platform key -- from a Colab secret read it we do (B3: NEVER hardcoded). With it, the
# News body comes FIRST from the Guardian API (raw bodyText in ONE ~0.2s call -- no browser, no consent
# wall), and only NON-Guardian stories fall back to the headless browser. ABSENT -> the Guardian fast-path
# simply skips, the browser handles News (just slower). A FREE key: open-platform.theguardian.com/access
try:
    from google.colab import userdata as _ud
    config.retrieval.guardian_api_key = _ud.get('guardian_key') or ''
except Exception:
    config.retrieval.guardian_api_key = config.retrieval.guardian_api_key or ''

print('mode:', config.mode)
print('competition_id:', config.game.competition_id, '| game_mode:', config.game.game_mode)
print('aim_seconds:', config.game.aim_seconds, '(below the 30s wall, a network margin this keeps)')
print('model:', config.model.name, '|', config.model.quantization)
print('Guardian API:', 'KEY SET (fast body path armed)' if config.retrieval.guardian_api_key else 'no key -> News uses browser fallback')

## 3 · Load + warm up the model
Identical to §2 — the cold-start cost paid **before** any timed question, so the 30s wall a cold load never eats.

In [ ]:
import time
from inference.engine import TransformersEngine

# Once, the model we load -- the cold-start cost, here we pay it.
t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine already in VRAM, skipping load.')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

# Warm up -- the first-call kernels, compiled before any timed question they are.
t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

## 4 · Wire the pipeline (same as offline) + log in to the game
DI exactly as in §2 (D-006) — only now a logged-in `GameClient` we also build.

Credentials: a PoliMi email as the username; the password from a **Colab secret** named
`poli-millionaire` we read (hardcode it, we must NOT — B3). Add it via the 🔑 panel on the left.

In [ ]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder, RoutingPromptBuilder
from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
from agent.pipeline import QAPipeline
from tools import default_tools, solve_maths
from retrieval import build_retriever

# Phase 4 RAG: the routing retriever, built from config ONLY when config.retrieval.enabled.
# Ablation (RAG on vs off) = the `enabled` flag. `source` picks the strategy:
#   "routed"    -> per question: News -> live web (Google News RSS + headless-Chromium body), else -> FAISS/Wikipedia.
#   "wikipedia" | "web" | "faiss" -> single-backend ablations.
# `needs_retrieval` still gates per question. News (post-cutoff) the live web NEEDS -- Wikipedia alone
# left News at 2/7, the breaking 2026 facts it cannot hold; Google News headlines + the article body we add.
retriever = build_retriever(config.retrieval)
print('config.retrieval:', config.retrieval)   # the LOADED flags -- if RAG is OFF, here enabled=False you see.
print('RAG:', (f'ON  source={config.retrieval.source}  top_k={config.retrieval.top_k}') if retriever else 'OFF')

# The collaborators, injected into the pipeline they are (D-006) -- identical to offline.
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
    classifier=QuestionClassifier(),
    retriever=retriever,       # Phase 4: RAW evidence; News->web, else->FAISS/Wikipedia; gated + graceful.
    tools=default_tools(),     # Phase 3 ON: the safe-AST calculator. ONLY on maths questions it fires
                               # (needs_calculator gates it) -- on Entertainment etc. a harmless no-op it is.
    latency_budget_s=config.latency_budget_s,
)

# --- Maths (comp 3): REVERTED to the 5bcf593 known-good config (Maths 9/10, reached level 9) ---
# After 5bcf593 we tried two "improvements" that both made Maths worse:
#   * 403b989 added the calculator as a match-gated verifier -- it helped on some Qs but introduced new
#     failure modes (e.g. Q6767 group-theory: calc emitted 4*12/(2*2)=12 mapping to wrong option).
#   * 5130bac switched to cot_maths_v1 with worked exemplars -- the exemplars anchored variable setup
#     (Q6777 chain now writes "Let s=speed, p=price") but the model still slipped on the algebra step
#     and answer-to-option mapping; net result was no better than cot_v2.
# Reverting to the cot_v2 + NO-calculator + single-pass config until we have evidence a change wins.
# cot_maths_v1 stays REGISTERED (no harm) but unused; the calculator stays in src but Maths skips it.
# ADAPTIVE ROUTING (offline experiment, §4): a RoutingPromptBuilder picks the prompt per
# question. CONSERVATIVE policy -- re-route ONLY the shapes we have evidence for to
# structured_enumeration_cot (interval-counting fixed 0.4->1.0, incl. the clock-chime death that capped
# Maths at level 9; temporal + discrete also helped); EVERYTHING ELSE (arithmetic, logic, concept/stats)
# stays on the battle-tested cot_v2 via the fallback. Minimal regression risk, targets the documented loss.
pipeline_maths = QAPipeline(
    engine=engine,
    prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
    classifier=QuestionClassifier(),
    retriever=None,            # Maths: NO retrieval -- it only distracts here.
    tools=None,                # NO calculator -- at n=1 it would clobber the chain on numeric Qs (run #8).
    solver=solve_maths,        # DETERMINISTIC type-specific solver -- short-circuits the LLM on solvable
                               # types (finite-field roots, gcd, characteristic, sum/product, reflection,
                               # triangle, %); abstains on all else (0 regressions on the logs).
    latency_budget_s=config.latency_budget_s,
    max_new_tokens=450,        # 30s WALL guard. Raised 300->450 post-B3 (2026-06-02): B3 keeps only SHORT
                               # time-interval questions in structured enumeration, so cot_v2's terse chains
                               # now finish in 5-10s (verified run: max 10.5s, mean 5.4s, zero turns >25s) --
                               # big headroom. 450 lets a legitimately long chain reach 'Answer:' rather than
                               # truncate. CAVEAT: ~16 tok/s => 450 tok ~= 28s, ABOVE the 25s aim -- only safe
                               # because chains rarely run that long now; dial back toward 400 if any Maths
                               # turn nears the wall. (latency_budget is advisory -- the token cap is the guard.)
    # self_consistency_n defaults to 1 -- SC dropped for Maths (run #8: it timed out, gave no benefit).
)
print('Maths pipeline (comp 3): ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2)'
      ' + 300 tokens (30s-wall safe) + single-pass (n=1) + NO retrieval + NO calculator')

# --- Per-race pipelines (D-006): EACH competition its OWN QAPipeline, so tuning ONE race never bleeds
# into another. The assignment ENCOURAGES per-topic strategies ("are certain models better at certain
# topics?") and rewards it on both the leaderboard and the investigation score -- and News must stay
# stable while we tune the rest. Today every race except Maths is the SAME few_shot_v1+RAG recipe, just
# as a separate instance -> behaviour identical now, independently tunable later (give a race its own
# strategy/retriever/tools and only that race changes).
#   * the heavy FAISS/Wikipedia retriever the four KNOWLEDGE races SHARE (one model load, not four);
#   * News gets its OWN web retriever instance, so its config can be tuned in isolation;
#   * Maths uses no retriever (pipeline_maths above).
knowledge_retriever = retriever                     # the routed retriever built above (FAISS/Wikipedia).
news_retriever = build_retriever(config.retrieval)  # News' OWN instance (web path; light, no model load).

def _make_race_pipeline(retr):
    """A fresh QAPipeline twin of the shared recipe -- few_shot_v1 + RAG + classifier + tools."""
    return QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
        classifier=QuestionClassifier(),
        retriever=retr,
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )

# --- Entertainment (comp 0): its OWN pipeline, `few_shot_entertainment` it runs (D-ENT) ---
# Entertainment is single-fact pop-culture recall (film / music / TV / books / games / sport). The shared
# few_shot_v1 primes the FORMAT with generic exemplars (capital-of, photosynthesis, 6x7) -- right shape,
# wrong register. `few_shot_entertainment` swaps those for film/music/TV exemplars + a domain-aware
# instruction, so the prime matches the questions asked and the small model reaches for the RIGHT kind of
# fact. NO chain-of-thought (recall drifts under CoT); RAG stays ON (Wikipedia/FAISS covers entertainment
# well and grounds the harder later-level facts), gated by `needs_retrieval`; the calculator is a harmless
# no-op here. Isolated from the other races -- editing this never touches News/Maths.
def _make_entertainment_pipeline(retr):
    return QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy="few_shot_entertainment"),
        classifier=QuestionClassifier(),
        retriever=retr,            # Phase 4: FAISS/Wikipedia evidence, gated per question.
        tools=default_tools(),     # no-op on entertainment (needs_calculator never fires here).
        latency_budget_s=config.latency_budget_s,
    )

pipeline_entertainment = _make_entertainment_pipeline(knowledge_retriever)
print('Entertainment pipeline (comp 0): few_shot_entertainment + RAG (FAISS/Wikipedia, gated) + no CoT')

RACE_PIPELINES = {
    0: pipeline_entertainment,                     # Entertainment (few_shot_entertainment + RAG)
    1: _make_race_pipeline(knowledge_retriever),   # Ancient History & Politics
    2: _make_race_pipeline(knowledge_retriever),   # Science & Nature
    3: pipeline_maths,                             # Maths (cot_v2 routing, NO retrieval, NO tools)
    4: _make_race_pipeline(knowledge_retriever),   # Philosophy & Psychology
    5: _make_race_pipeline(news_retriever),        # News (its OWN web retriever)
}

def pipeline_for(cid):
    """The pipeline for competition `cid` -- each race its own, so per-race tuning stays isolated."""
    return RACE_PIPELINES.get(cid, pipeline)

print('Per-race pipelines wired:',
      {c: ('entertainment' if c == 0 else 'maths' if c == 3 else 'news' if c == 5 else 'knowledge') for c in range(6)})

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')       # <-- your PoliMi email, fill it you must.
PASSWORD = userdata.get('password')  # <-- a Colab secret, NOT hardcoded (B3).

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions and their ids, list them we do (safe -- no timer this starts).
for c in game_client.list_competitions():
    ml = getattr(c, 'max_levels', '?')
    print('  id=', c.id, '|', c.name, '| max_levels=', ml)

## 5 · ▶ Play ONE real game  (consumes a leaderboard attempt + starts the 30s timer)
`run_session` sees `mode='live'` and drives `LiveRunner`: start game → our pipeline answers each
question → submit by integer `Option.id` → log one `EvalRecord` per turn (with `correct` from the server).

⚠️ Deliberately run this. Each turn prints live; the full per-turn record lands in the JSONL log.

In [ ]:
from evaluation.runner import run_session

# The ONE switch -- mode='live' it sees, so LiveRunner over the real game it drives.
# Per-race: the single game uses the pipeline for the chosen competition (config.game.competition_id).
run_path = run_session(
    pipeline_for(config.game.competition_id),
    config,
    game_client=game_client,
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
)
print('Live run written to:', run_path)

## 6 · Results — how far did we get?
From the same JSONL log as offline it derives. `correct` is None where the server withheld it
(e.g. a timeout). Highest level reached + a per-turn table, here we show.

In [ ]:
from evaluation.metrics import load_runs

# The live run, back into a DataFrame we read.
df = load_runs([run_path])


def _run_reached(df):
    """The level THIS run CLIMBED to -- the server's `reached_level` telemetry (from AnswerResult),
    its max across the turns. NOT `level` (that is the per-turn rung the server sends as 0 -- the old
    bug, where 'Highest level reached: 0' it always printed). Fallbacks: current_level, then level.
    None, only when the server told us nothing at all."""
    for col in ('reached_level', 'current_level', 'level'):
        if col in df.columns:
            s = df[col].dropna()
            if not s.empty:
                return int(s.max())
    return None


answered = len(df)
known = df[df['correct'].notna()]
n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
acc = (n_correct / len(known)) if len(known) else float('nan')
reached = _run_reached(df)   # FIXED: the server's reached_level, not the always-0 per-turn `level`.

print(f'Questions answered: {answered}')
print(f'Correct: {n_correct}/{len(known)}  (accuracy {acc:.0%})')
print(f'Highest level reached: {reached}')

# Per question, the full card -- the choices too, so the wrong ones inspect we can
# (the picked letter, by its option text now we read). Note: `options` only the live
# runs from NOW ON carry; an older log an empty cell shows.
for _, row in df.iterrows():
    opts = row['options'] if 'options' in df.columns and isinstance(row['options'], dict) else {}
    picked = row['predicted_answer']
    mark = {True: 'CORRECT', False: 'WRONG'}.get(row['correct'], 'UNKNOWN')  # None (server withheld) it is.
    print('\n' + '=' * 70)
    print(f"[{mark}] qid={row['qid']} | reached_level={row.get('reached_level')} | topic={row['topic']} | latency={row['latency_s']:.1f}s")
    print(f"Q: {row['question_text']}")
    if opts:
        for letter, text in opts.items():
            arrow = '   <-- model picked' if letter == picked else ''
            print(f"   {letter}. {text}{arrow}")
    else:
        print(f"   (options not logged for this run)  model picked: {picked}")
    print(f"correct={row['correct']}  |  confidence={row['confidence']}")


## 7 · Observations

What the live run confirms (full per-competition results in the sweep below):

- **The API contract holds.** Submitting by `Option.id` and reading the server's `reached_level` work as
  documented — the per-run summary above reports the level actually climbed, not the always-0 per-turn `level`.
- **Latency stays under the wall** once network RTT is added: the heaviest turns (Maths `cot_v2`) finish
  well inside 30 s; no turn nears the limit.
- **Per-race routing takes effect live:** the diagnostics table prints `prompt_strategy` / `retrieval_used`
  / `tool_used` per question, so each competition is verifiably running its own pipeline.
- **Live tracks offline:** the knowledge races hold near the ~87 % the dev set predicted; the gap shows up
  exactly where expected — Maths reasoning and post-cutoff News.

**Switching modes is a one-liner:** offline ⇄ live is just `config.mode` + whether you pass `game_client`.
For a quick offline re-check: `run_session(pipeline, RunConfig.from_yaml('configs/base.yaml'))` — no game_client.

> Be polite to the server (rate limits are real).


## 8 · ▶ Sweep ALL competitions (live) + scores + every wrong question

Plays ONE live game in **each** of the 6 competitions (0–5), back to back, with a polite pause between
(the assignment asks: avoid rapid consecutive requests). Each competition logs its own run dir
(`live_comp{id}`). Then a per-competition **scoreboard** + the consolidated list of **every wrong
question** — text, options, and the letter our model picked — also saved to `experiments/wrong_questions.jsonl`.

> ⚠️ This plays **6 REAL games** (timers + leaderboard). Run it deliberately. **Maths (comp 3)** runs its
> own pipeline — `cot_v2` (CoT + option-matching check), **single-pass** (self-consistency was dropped after
> run #8 timed out at 41s), no retrieval, no calculator; every other competition uses the shared `few_shot_v1`
> pipeline. **RAG (Phase 4) is ON** when `configs/live.yaml` has `retrieval.enabled: true` —
> `source: "routed"` sends **News → live web (DuckDuckGo)** and every other topic → **FAISS corpus /
> Wikipedia**. (`needs_retrieval` still gates it per question.) One game failing (e.g. a rate-limit) won't
> abort the rest — it's logged as a blank row and the sweep continues.

This is the multi-competition counterpart of the single-game run above.


In [ ]:
from evaluation.runner import run_all_competitions

# Clear THIS sweep's prior run dirs FIRST. The logger appends within a run and the dir name is reused
# (live_comp{id}), so without this a re-run piles onto the previous one -> duplicate qids, inflated counts,
# mixed strategies. This is a SHELL command (not cached Python), so it works even if the logger's
# truncate-on-open fix hasn't loaded yet (that needs a kernel Restart; this rm does not).
!rm -rf {REPO_ROOT}/experiments/runs/live_comp*
print('cleared prior live_comp* run dirs')

COMP_NAMES = {0: 'Entertainment', 1: 'Ancient History & Politics', 2: 'Science & Nature',
              3: 'Maths', 4: 'Philosophy & Psychology', 5: 'News'}

# One live game in EVERY competition (0-5), back to back, a polite pause between (PDF: no rapid requests).
# Each competition its own run dir gets (live_comp{id}); one game's failure the rest never sinks.
# Maths (comp 3) its OWN pipeline gets via `pipeline_for` -- cot_v2 + single-pass (n=1) + NO retrieval + NO
# calculator (run #8: cot_v2 + self-consistency timed out at 41s on a question it answered CORRECTLY; SC gave
# no benefit, so dropped). Routed by competition_id, the reliable LIVE signal; every other competition the
# shared few_shot + RAG `pipeline` keeps.
comp_runs = run_all_competitions(
    pipeline, config, game_client,
    competition_ids=range(6),
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
    pause_s=8.0,
    on_competition=lambda cid: print(f"\n===== ▶ Competition {cid}: {COMP_NAMES.get(cid, '?')} ====="),
    pipeline_for=pipeline_for,   # per-race: each competition its own pipeline (Maths/News/knowledge)
)
print('\nSweep done. Run paths:')
for cid, path in comp_runs:
    print(f"  comp {cid} {COMP_NAMES.get(cid, ''):28} -> {path}")

In [ ]:
import json
import pandas as pd
from evaluation.metrics import load_runs


def _retr_docs(r):
    """The retrieved doc ids as a clean list, defensively we read (NaN/None -> [])."""
    v = r.get('retrieved_doc_ids')
    return list(v) if isinstance(v, (list, tuple)) else []


def _run_reached(df):
    """The level THIS run CLIMBED to -- the server's `reached_level` telemetry (from AnswerResult),
    its max across the turns. NOT `level` (the per-turn rung the server sends as 0 -- the old bug that
    made this column always 0). Fallbacks: current_level, then level. None only if the server said nothing."""
    for col in ('reached_level', 'current_level', 'level'):
        if col in df.columns:
            s = df[col].dropna()
            if not s.empty:
                return int(s.max())
    return None


def _lb_entry(cid):
    """The AUTHORITATIVE leaderboard row for us in this competition (reached_level + score) -- the REAL
    scored metric (D-014). Crash-safe: None on any error (player not found, client shape differs, ...)."""
    try:
        me = game_client._client.user.username
        return game_client._client.leaderboard.find_player(cid, me)
    except Exception:
        return None


rows, wrong_all, usage = [], [], []
for cid, path in comp_runs:
    lb = _lb_entry(cid)
    lb_level = getattr(lb, 'reached_level', None)
    lb_score = getattr(lb, 'score', None)
    if path is None:   # That competition failed (e.g. rate-limited) -- a blank row it gets (lb still shown).
        rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': 0,
                     'correct': 0, 'of': 0, 'accuracy': float('nan'),
                     'run_reached': None, 'lb_level': lb_level, 'lb_score': lb_score})
        continue
    df = load_runs([path])
    known = df[df['correct'].notna()]
    n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
    acc = (n_correct / len(known)) if len(known) else float('nan')
    rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': len(df),
                 'correct': n_correct, 'of': len(known), 'accuracy': acc,
                 # run_reached = THIS sweep's climb (server telemetry); lb_level = all-time scored best.
                 'run_reached': _run_reached(df), 'lb_level': lb_level, 'lb_score': lb_score})
    # Per-competition tool/retrieval usage -- the diagnostic the wrong-dump was MISSING.
    # retrieval_fired = needs_retrieval gated ON; docs_landed = snippets actually came back
    # (fired but 0 docs => the backend returned nothing, e.g. DuckDuckGo blocked on the Colab IP).
    n_retr = int(df['retrieval_used'].fillna(False).astype(bool).sum()) if 'retrieval_used' in df else 0
    n_docs = int(sum(len(_retr_docs(r)) > 0 for _, r in df.iterrows())) if 'retrieved_doc_ids' in df else 0
    n_tool = int(df['tool_used'].notna().sum()) if 'tool_used' in df else 0
    usage.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': len(df),
                  'retrieval_fired': n_retr, 'docs_landed': n_docs, 'tool_calls': n_tool})
    for _, r in df[df['correct'] == False].iterrows():   # correct is None (timeout) -> excluded, good.
        wrong_all.append((cid, COMP_NAMES.get(cid, ''), r))

# --- Scoreboard, per competition + overall ---
# run_reached = how far THIS sweep climbed (server's reached_level telemetry, NOT the always-0 per-turn rung).
# lb_level / lb_score = the AUTHORITATIVE leaderboard numbers (all-time best -- what we are actually scored on).
summary = pd.DataFrame(rows)
print('SCORES BY COMPETITION')
print(summary.to_string(index=False))
tot_c, tot_n = int(summary['correct'].sum()), int(summary['of'].sum())
print((f"\nOVERALL: {tot_c}/{tot_n} = {tot_c / tot_n:.1%}") if tot_n else "\nOVERALL: no graded answers")

# --- RAG / tool usage by competition (the News path, here we finally SEE it) ---
print('\nRAG / TOOL USAGE BY COMPETITION')
print(pd.DataFrame(usage).to_string(index=False))

# --- Every wrong question: text + options + the letter our model picked + WHY (tools/retrieval) ---
print(f"\n{'=' * 72}\nEVERY WRONG QUESTION  ({len(wrong_all)})\n{'=' * 72}")
for cid, name, r in wrong_all:
    opts = r['options'] if 'options' in r and isinstance(r['options'], dict) else {}
    picked = r['predicted_answer']
    docs = _retr_docs(r)
    print(f"\n[comp {cid} · {name}] qid={r['qid']} reached_level={r.get('reached_level')}")
    print(f"Q: {r['question_text']}")
    for letter, text in opts.items():
        mark = '   <-- our pick (WRONG)' if letter == picked else ''
        print(f"   {letter}. {text}{mark}")
    # The diagnostic line -- did a tool fire? did retrieval fire, and did snippets LAND?
    # tool=None + retrieval_used=False  => the plain model answered (no help asked for).
    # retrieval_used=True + docs_landed=0 => the retriever fired but came back EMPTY (the News bug to watch).
    print(f"   -> tool={r.get('tool_used')} | retrieval_used={bool(r.get('retrieval_used'))} | "
          f"docs_landed={len(docs)}" + (f" {docs[:3]}" if docs else ""))

# --- Save a clean consolidated record of the wrong questions (now WITH the tool/retrieval trace) ---
out = os.path.join(REPO_ROOT, 'experiments', 'wrong_questions.jsonl')
with open(out, 'w', encoding='utf-8') as f:
    for cid, name, r in wrong_all:
        f.write(json.dumps({
            'competition_id': cid, 'competition': name, 'qid': r['qid'],
            'level': r['level'], 'reached_level': r.get('reached_level'),
            'question_text': r['question_text'],
            'options': r['options'] if isinstance(r.get('options'), dict) else {},
            'our_wrong_pick': r['predicted_answer'],
            'tool_used': r.get('tool_used'),
            'retrieval_used': bool(r.get('retrieval_used')),
            'retrieved_doc_ids': _retr_docs(r),
        }, ensure_ascii=False) + '\n')
print(f"\nWrong questions saved -> {out}  ({len(wrong_all)} rows)")


In [ ]:
import json, collections
from pathlib import Path

# Per-question tool/retrieval/STRATEGY detail across ALL competitions. The diagnostics:
#   - retr=True + docs=0  => the retriever fired but came back EMPTY (DDG blocked / Wikipedia miss).
#   - strat=cot_v1 on comp 3 ONLY  => the Maths routing (pipeline_for) took effect; few_shot_v1 elsewhere.
# Logger is now truncate-per-run (one sweep = one fresh records.jsonl), so these rows are THIS run only.
for cid, path in comp_runs:
    if path is None:
        continue
    p = Path(path) / "records.jsonl"
    if not p.exists():
        continue
    rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"\n===== comp {cid} {COMP_NAMES.get(cid, '')}  (n={len(rows)}) =====")
    print("  tool_used:      ", collections.Counter(r.get("tool_used") for r in rows))
    print("  retrieval_used: ", collections.Counter(r.get("retrieval_used") for r in rows))
    print("  prompt_strategy:", collections.Counter(r.get("prompt_strategy") for r in rows))
    for r in rows:
        docs = r.get("retrieved_doc_ids") or []
        flag = '' if r.get('correct') else '  <-- WRONG'
        print(f"  {r['qid']:>6} strat={str(r.get('prompt_strategy')):<12} "
              f"tool={str(r.get('tool_used')):<11} retr={str(r.get('retrieval_used')):<5} "
              f"docs={len(docs):<2} correct={str(r.get('correct')):<5} | {r['question_text'][:50]}{flag}")


---
# §9 · Conclusions — every investigation question → our finding

| Investigation question | Where | Our finding |
|---|---|---|
| **30 s feasibility** | §2 | Yes. Baseline median ~0.91 s, p95 ~1.0 s, 0 budget violations on a T4. Even the heaviest live turn (Maths `cot_v2`, ~20 s) stays under the 30 s wall. |
| **Per-topic strengths** | §2 | Strong on factual recall (History / Entertainment / News / Philosophy ≈ 100% at baseline); weak on **Maths (50%)** and **Science (75%)** — reasoning/calculation-bound, not recall-bound. |
| **Overconfidence** | §2 | Yes — poorly calibrated: confidently wrong answers reported `confidence = 1.0` (e.g. the moons question). |
| **Failure modes** | §2, §4 | Mostly genuine reasoning/knowledge gaps, not parser bugs. §4's taxonomy adds *overthinking* (recall Qs over-reasoned) vs *option-matching slips* (right derivation, wrong letter). |
| **Best prompt (zero / few / CoT)** | §3 | No single winner — few-shot helps factual recall, CoT helps reasoning; `cot_v2` fixes the Maths option-matching slip. This motivates §4. |
| **Prompt sensitivity / adaptive prompting** | §4 | **Adaptive routing beats every fixed prompt** — no one prompt is best across categories. Structured enumeration *cures* interval-counting/temporal Qs but *hurts* factual recall; checklists win on logic/multi-hop; direct answering wins on factual/commonsense. |
| **RAG lift** | §6 | Net positive, especially News (the model's prior is often wrong on post-cutoff facts). Raw-content-only sources; gated so it doesn't fire on pure-reasoning Qs. Ablation in §6. |
| **Calculator lift** | §5 | Built + evaluated, but **not active in the final run**: `needs_calculator` only fires on arithmetic Qs, which only occur in Maths — and Maths sets `tools=None` (at n=1 the tool clobbered correct `cot_v2` chains). A no-op on the other races. |
| **Ensemble reliability** | §4, §5 | Implemented + evaluated, but **self-consistency is off everywhere** (`n=1`): on Maths SC(n=3) blew the 30 s wall (~41 s) and all chains shared the same slip, so it could not self-correct. Single-pass `cot_v2` is the live choice. |
| **Model A/B · size vs quality** | §7 | Base 7B vs a math-specialised open model on the Maths race (accuracy vs latency trade-off). |
| **Live leaderboard result** | §8 | Entertainment / History / Science / Philosophy reach **level 15 (max)**; **Maths level 11**; **News ~level 12** (~82–85% per question) — confirm against the fresh §8 sweep. |

### Honest limitations
- The ~7B model has a knowledge ceiling (~55–65%) on the hardest knowledge rungs; RAG narrows but does not close it.
- Heavy retrieval bursts can trigger source rate-limits; we degrade gracefully (empty context, never a crashed turn).
- Self-consistency is unused live purely for latency; it remains valuable offline (see §4).